# Penalties Analysis

Common setup and helpers are defined once here.

Data: auto-loads every `{chain}_*.csv` in `PENALTIES_DATA_DIR` and groups by `blockchain`. Cross-chain views activate automatically once >1 chain is present.

In [ ]:
# === COMMON SETUP & HELPERS (defined once) ===
import glob, os, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dune_client.client import DuneClient
from dune_client.query import QueryBase
from dune_client.types import QueryParameter
from dotenv import load_dotenv
from dotenv import dotenv_values
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import urllib.request, zipfile
import subprocess

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 80)
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True, 'grid.alpha': 0.3})
DATA_DIR = os.environ.get('PENALTIES_DATA_DIR', '.')
NIL = {'<nil>': np.nan, 'None': np.nan, 'nan': np.nan, '': np.nan}
WEI = 1e18

NUMERIC = ['volume_native','slippage_native','reward_penalty_native','reward_penalty_uncapped_native',
    'reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native','reward_cap_upper_native',
    'reference_score','observed_score','execution_cost_native','slippage_usd','order_size_usd','markout_usd',
    'markout_relative','slippage_tolerance_bps','calculated_slippage_bps','seconds_since_created',
    'seconds_to_settle','auction_id','settlement_block','block_deadline','executed_sell']  
BOOL = ['settled','is_excluded_from_penalties','is_quote_reward_eligible','partially_fillable',
    'is_out_of_market','smart_slippage']
# whole-token (/WEI) THEN USD (avoids 1e18 blowup with a per-token price)
ECON_NATIVE = ['reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native',
    'reward_penalty_native','reward_cap_upper_native','execution_cost_native','reference_score','observed_score']

def _num(s): return pd.to_numeric(s.astype(str).replace(NIL), errors='coerce')

def load_chain_csvs(data_dir='.', chains=None):
    """Load only chain data CSVs whose filename starts with a known chain name."""
    if chains is None:
        chains = [
            'arbitrum',
            'avalanche_c',
            'base',
            'bnb',
            'ethereum',
            'gnosis',
            'polygon',
        ]

    prefixes = tuple(f'{ch}_' for ch in chains)
    paths = sorted(
        p for p in glob.glob(os.path.join(data_dir, '*.csv'))
        if os.path.basename(p).startswith(prefixes)
    )
    if not paths:
        raise FileNotFoundError(f'no chain CSVs found in {data_dir!r} for chains={chains}')
    frames = []
    for p in paths:
        d = pd.read_csv(p, low_memory=False)
        d['_source_file'] = os.path.basename(p)

        md5 = hashlib.md5(open(p, 'rb').read()).hexdigest()[:12]
        print(f'  {os.path.basename(p):45s} rows={len(d):>7,}  md5={md5}')

        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def native_token_usd_price(df):
    """Per-chain-day median native-token USD price from settled rows (order_size_usd / volume_tok),
    applied to all rows incl reverts. Stable vs noisy per-row slippage ratios. Falls back to per-chain."""
    s = df['settled'] == True
    vol_tok = df['volume_native'] / WEI
    px = (df['order_size_usd'] / vol_tok).where(s & (vol_tok > 0))
    day = pd.to_datetime(df['auction_timestamp'], errors='coerce').dt.date
    key = pd.DataFrame({'chain': df['blockchain'], 'day': day, 'px': px})
    cd = key.groupby(['chain', 'day'])['px'].transform('median')
    ch = key.groupby('chain')['px'].transform('median')
    return cd.fillna(ch)

def prepare(df):
    """Coerce types, derive native→USD price, build *_tok/*_usd economic cols, two P&L concepts, bps, flags.
    Grains: attempt = (auction_id, order_uid); reward/penalty = (auction_id, solver); markout/size_usd = settled-only."""
    df = df.copy()
    for c in NUMERIC:
        if c in df: df[c] = _num(df[c])
    for c in BOOL:
        if c in df: df[c] = df[c].astype(str).str.lower().map({'true': True, 'false': False})
    df['native_usd_price'] = native_token_usd_price(df)             # USD per whole token
    for c in ECON_NATIVE:                                           # wei -> token -> USD
        base = c[:-len('_native')] if c.endswith('_native') else c
        df[base + '_tok'] = df[c] / WEI
        df[base + '_usd'] = df[base + '_tok'] * df['native_usd_price']
    df['volume_tok'] = df['volume_native'] / WEI
    df['volume_usd'] = df['volume_tok'] * df['native_usd_price']
    # size_usd: Dune order_size_usd is settled-only; fall back to volume_usd (present on reverts)
    df['size_usd'] = df['order_size_usd'].where(df['order_size_usd'].notna(), df['volume_usd'])
    # GROSS excludes reward/penalty. Using when penalty/cap is the regressor (else tautological).
    # NET includes them. Realised welfare under the mechanism.
    df['gross_execution_pnl_usd'] = np.where(df['settled'] == True, df['slippage_usd'] - df['execution_cost_usd'], np.nan)
    df['net_mechanism_pnl_usd'] = (df['reward_penalty_usd'].fillna(0) + df['slippage_usd'].fillna(0) - df['execution_cost_usd'].fillna(0))
    df['reverted'] = ~df['settled'].fillna(False)
    df['cap_binds'] = df['penalty_uncapped_native'] > df['penalty_cap_native']
    df['is_solo_bidder'] = df['reference_score'] == 0              # reference_score == 0 = penalty structurally 0
    df['markout_bps'] = df['markout_relative'] * 1e4
    df['penalty_bps'] = (df['penalty_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    df['reward_bps'] = (df['reward_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    # DATA-QUALITY GUARDS: score-degenerate: reverted rows with observed_score=0 & huge reference_score driving an impossible uncapped/cap ratio
    _put = df['penalty_uncapped_native'] / WEI
    _pct = (df['penalty_cap_native'] / WEI).replace(0, np.nan)
    _flag_score = (df['reverted'] & (df['reference_score'] > 0)
                   & (df['observed_score'] == 0) & ((_put / _pct) > 1e4)).fillna(False)

    _bad = _flag_score
    if _bad.any():
        print(f'data-quality: dropping {_bad.sum()} score-degenerate rows; '
              f'max penalty_uncapped_native = {df.loc[_bad, "penalty_uncapped_native"].max():.2e}')
    df = df[~_bad].copy()
    return df

def apply_filters(df, fok_only=True, in_market_only=True, exclude_penalty_excluded=True):
    """In-market fill-or-kill only; reverted winners KEPT (the signal). Returns (df, funnel)."""
    funnel = {'start': len(df)}; out = df
    if fok_only: out = out[out['partially_fillable'] == False]; funnel['fill_or_kill'] = len(out)
    if in_market_only: out = out[out['is_out_of_market'] == False]; funnel['in_market'] = len(out)
    if exclude_penalty_excluded: out = out[out['is_excluded_from_penalties'] == False]; funnel['penalty_eligible'] = len(out)
    funnel['reverted_kept'] = int(out['reverted'].sum()); funnel['settled_kept'] = int((~out['reverted']).sum())
    return out.copy(), funnel

def reward_per_auction_solver(df):
    """Aggregate to (chain, auction, solver). Reward/penalty repeat across a multi-order solution's
    rows ('first'); order-level economics (gross pnl, volume) are summed."""
    g = df.groupby(['blockchain', 'auction_id', 'solver'], dropna=False)
    out = g.agg(
        solver_name=('solver_name', 'first'),
        reward_usd=('reward_usd', 'first'),
        penalty_usd=('penalty_usd', 'first'),
        penalty_uncapped_usd=('penalty_uncapped_usd', 'first'),
        reward_penalty_usd=('reward_penalty_usd', 'first'),
        penalty_cap_usd=('penalty_cap_usd', 'first'),
        gross_execution_pnl_usd=('gross_execution_pnl_usd', 'sum'),
        volume_usd=('volume_usd', 'sum'),
        size_usd=('size_usd', 'sum'),
        reverted=('reverted', 'any'),
        cap_binds=('cap_binds', 'any'),
        order_count=('order_uid', 'nunique'),
    ).reset_index()
    out['net_mechanism_pnl_usd'] = out['reward_penalty_usd'].fillna(0) + out['gross_execution_pnl_usd'].fillna(0)
    out['penalty_bps'] = (out['penalty_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    out['reward_bps'] = (out['reward_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    return out

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z**2/n; c = p + z**2/(2*n)
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return (p, (c-h)/d, (c+h)/d)

FIXED_BINS = [0, 100, 1_000, 10_000, 100_000, np.inf]             
FIXED_LABELS = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']

def add_fixed_buckets(df, col='size_usd'):
    df = df.copy()
    df['size_fixed'] = pd.cut(df[col], bins=FIXED_BINS, labels=FIXED_LABELS, include_lowest=True)
    return df

def add_chain_quantile_buckets(df, value='size_usd', q=5):
    """Per-chain quantile buckets (primary size view). duplicates='drop' + ordinal relabel so tie-heavy
    chains don't crash on fixed labels; ordered categorical so Q10 doesn't sort before Q2."""
    out = df.copy(); out['size_q'] = pd.Series(pd.NA, index=out.index, dtype='object')
    for ch, idx in out.groupby('blockchain').groups.items():
        s = out.loc[idx, value]; valid = s[s.notna() & np.isfinite(s) & (s > 0)]
        if len(valid) < q: continue
        try: binned = pd.qcut(valid, q=q, duplicates='drop')
        except ValueError: continue
        out.loc[valid.index, 'size_q'] = binned.cat.codes.map(lambda c: f'Q{c+1}')
    out['size_q'] = pd.Categorical(out['size_q'], categories=[f'Q{i}' for i in range(1, q+1)], ordered=True)
    return out

def ecdf(ax, series, label, color=None):
    x = np.sort(pd.Series(series).dropna().values)
    if len(x): ax.plot(x, np.arange(1, len(x)+1)/len(x), label=label, color=color)

def barpos(ax, labels, vals, **kw):
    """Bar with explicit numeric x + string labels (avoids matplotlib categorical-float errors)."""
    x = np.arange(len(labels)); ax.bar(x, np.asarray(vals, float), **kw)
    ax.set_xticks(x); ax.set_xticklabels([str(l) for l in labels])

print('helpers defined')

In [ ]:
# === SUBSET CONTROL ===========================================================
SUBSET_MODE = False
# ADD CHAIN NAMES HERE FOR SUBSET ANALYSIS
SUBSET_CHAINS = ['polygon', 'bnb', 'avalanche_c']
SUBSET_PAIR = ('ETH', 'USDC')                 
SUBSET_SOLVER_SCOPE = SUBSET_CHAINS

# Canonical WETH / USDC addresses per chain.
# ADD TOKEN PAIRS HERE FOR SUBSET ANALYSIS
SUBSET_TOKENS = {
    'polygon': {
        'ETH':  {'0x7ceB23fD6bC0adD59E62ac25578270cFf1b9f619', '0x11CD37bb86F65419713f30673A480EA33c826872', '0xe50fA9b3c56FfB159cB0FCA61F5c9D750e8128c8', '0x1280830F690D0E65195B3c61b028244C3A49f26D'},   
        'USDC': {'0x3c499c542cEF5E3811e1192ce70d8cC03d5c3359', '0x625E7708f30cA75bfd92586e17077590C60eb4cD', '0x750e4C4984a9e0f12978eA6742Bc1c5D248f40ed', '0x2791Bca1f2de4661ED88A30C99A7a9449Aa84174', '0x4318CB63A2b8edf2De971E2F17F77097e499459D'},    
    },
    'bnb': {
        'ETH':  {'0x2170Ed0880ac9A755fd29B2688956BD959F933F8','0x4DB5a66E937A9F4473fA95b1cAF1d1E1D62E29EA'},   
        'USDC': {'0x8ac76a51cc950d9822d68b83fe1ad97b32cd580d', '0x4268B8F0B87b6Eae5d897996E6b845ddbD99Adf3'},   
    },
    'avalanche_c': {
        'ETH':  {'0x49D5c2BdFfac6CE2BFdB6640F4F80f226bc10bAB', '0x8b82A291F83ca07Af22120ABa21632088fC92931', '0xe50fA9b3c56FfB159cB0FCA61F5c9D750e8128c8'},   
        'USDC': {'0xb97ef9ef8734c71904d8002f8b6bc66dd9c48a6e', '0x625E7708f30cA75bfd92586e17077590C60eb4cD', '0xA7D7079b0FEaD91F3e65f86E8915Cb59c1a4C664', '0x625E7708f30cA75bfd92586e17077590C60eb4cD', '0xe1bFC96d95BAdCB10Ff013Cb0C9C6c737ca07009'},    
    },
}

def _canonical_token_col(df, addr_col, chains_tokens=SUBSET_TOKENS):
    addr = df[addr_col].astype(str).str.lower()
    out = pd.Series(np.nan, index=df.index, dtype='object')
    for ch, toks in chains_tokens.items():
        chain_mask = df['blockchain'] == ch
        for sym, addrs in toks.items():
            out[chain_mask & addr.isin({a.lower() for a in addrs})] = sym
    return out

def subset_intersection(df,
                        pair=SUBSET_PAIR,
                        chains=SUBSET_CHAINS,
                        solver_scope=SUBSET_SOLVER_SCOPE):
    d = df.copy()
    a, b = pair[0].upper(), pair[1].upper()

    d['_sell_sym'] = _canonical_token_col(d, 'sell_token')
    d['_buy_sym']  = _canonical_token_col(d, 'buy_token')
    pair_mask = (((d['_sell_sym'] == a) & (d['_buy_sym'] == b)) |   
                 ((d['_sell_sym'] == b) & (d['_buy_sym'] == a)))    
    scope_src = d[d['blockchain'].isin(solver_scope)]
    n_chains = scope_src.groupby('solver')['blockchain'].nunique()
    keep_solvers = n_chains[n_chains >= len(solver_scope)].index

    d = d[pair_mask & d['blockchain'].isin(chains) & d['solver'].isin(keep_solvers)]
    return d.drop(columns=['_sell_sym', '_buy_sym']).copy()

if SUBSET_MODE:
    MIN_REVERTS, MIN_SETTLED = 15, 15                 # Section 10 solver-level
    MIN_REVERTS_CHAIN, MIN_SETTLED_CHAIN = 10, 10     # Section 10 chain-level
    MIN_PTS_LOGBIN, MIN_PTS_PER_BIN, MIN_PTS_DECILE = 15, 8, 40   
else:
    MIN_REVERTS, MIN_SETTLED = 50, 50
    MIN_REVERTS_CHAIN, MIN_SETTLED_CHAIN = 20, 20
    MIN_PTS_LOGBIN, MIN_PTS_PER_BIN, MIN_PTS_DECILE = 50, 20, 200

print(f'subset control defined (SUBSET_MODE={SUBSET_MODE})')

In [ ]:
print('loading:'); RAW = prepare(load_chain_csvs())
if SUBSET_MODE:
    n0 = len(RAW)
    RAW = subset_intersection(RAW)
    print(f'SUBSET MODE: {SUBSET_PAIR[0]}-{SUBSET_PAIR[1]} x {SUBSET_CHAINS}, '
          f'solvers on all {len(SUBSET_SOLVER_SCOPE)} — {n0:,} → {len(RAW):,} rows')
    if len(RAW) == 0:
        print('!! 0 rows, check token addresses against RAW[...].value_counts() per chain')
    else:
        print('solvers kept:', sorted(RAW['solver_name'].dropna().unique().tolist()))
        print('per-chain rows:', RAW.groupby('blockchain').size().to_dict())
DF, FUNNEL = apply_filters(RAW)
print('\nfunnel:', {k: f'{v:,}' for k, v in FUNNEL.items()})
print('chains:', DF['blockchain'].unique().tolist())
MULTI = DF['blockchain'].nunique() > 1
if not MULTI: print('single chain')

Sanity panel before interpretation: per-chain coverage + solver HHI, null-on-revert check for settled-only columns, and p50/p99/max to catch native-unit / huge-size artifacts that dominate totals.

In [ ]:
def coverage_table(df):
    rows = []
    for ch, d in df.groupby('blockchain'):
        shares = d['solver'].value_counts(normalize=True)
        rows.append({'chain': ch, 'attempts': len(d), 'orders': d['order_uid'].nunique(),
            'solvers': d['solver'].nunique(), 'revert_rate': d['reverted'].mean(),
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'median_size_usd': d['size_usd'].median(),
            'solver_hhi': float((shares**2).sum()), 'top_solver_share': float(shares.iloc[0]),
            'token_pairs': (d['sell_token'].astype(str)+'/'+d['buy_token'].astype(str)).nunique()})
    return pd.DataFrame(rows).set_index('chain')

def missingness_check(df):
    cols = ['order_size_usd','markout_usd','markout_relative','seconds_to_settle','slippage_usd','execution_cost_native']
    rev, st = df[df['reverted']], df[~df['reverted']]
    return pd.DataFrame({'null_on_revert_%': [100*rev[c].isna().mean() for c in cols],
                         'null_on_settled_%': [100*st[c].isna().mean() for c in cols]}, index=cols).round(1)

def outlier_diag(df):
    cols = ['volume_tok','penalty_uncapped_tok','size_usd','penalty_uncapped_usd']
    return {ch: pd.DataFrame({c: [d[c].quantile(.5), d[c].quantile(.99), d[c].max()] for c in cols},
                             index=['p50','p99','max']) for ch, d in df.groupby('blockchain')}
    
def data_quality_report(raw):
    """Report-only: which rows the prepare() guards flagged, per chain. Runs on RAW (pre-filter)."""
    d = raw.copy()
    d['_day'] = pd.to_datetime(d['auction_timestamp'], errors='coerce').dt.date
    _ratio = d['volume_native'] / d['executed_sell'].replace(0, np.nan)
    _ref = (d.assign(_r=_ratio).groupby(['sell_token', '_day'])['_r'].transform('median')
              .fillna(d.assign(_r=_ratio).groupby('sell_token')['_r'].transform('median')))
    d['implied_deviation'] = _ratio / _ref
    flag_price = ((d['implied_deviation'] > 100) | (d['implied_deviation'] < 1/100)).fillna(False)

    put = d['penalty_uncapped_native'] / WEI
    pct = (d['penalty_cap_native'] / WEI).replace(0, np.nan)
    flag_score = (d['reverted'] & (d['reference_score'] > 0)
                  & (d['observed_score'] == 0) & ((put / pct) > 1e4)).fillna(False)
    print('=== data-quality flags (report-only, on RAW) ===')
    print(f'  implied-price (>100x token-day median): {flag_price.sum()} rows')
    print(f'  score-degenerate:        {flag_score.sum()} rows')
    print(f'  either:                                 {(flag_price|flag_score).sum()} '
          f'({100*(flag_price|flag_score).mean():.4f}%)')
    bad = d[flag_price | flag_score]
    if len(bad):
        summary = bad.assign(flag_price=flag_price[bad.index], flag_score=flag_score[bad.index]).groupby('blockchain').agg(
            n_flagged=('auction_id', 'size'),
            n_price_flag=('flag_price', 'sum'),
            n_score_flag=('flag_score', 'sum'),
            max_uncapped_native=('penalty_uncapped_native', 'max'))
        display(summary)
    return bad

DQ_FLAGGED = data_quality_report(RAW)
print('=== coverage ==='); display(coverage_table(DF))
print('=== missingness (settled-only cols ~100% null on revert) ==='); display(missingness_check(DF))
print('=== outliers (p50/p99/max) ===')
for ch, t in outlier_diag(DF).items(): print(ch); display(t)

## 1. Revert rates by chain vs rewards & penalties
Per-chain revert rate (attempt grain, Wilson CI) with median reward/penalty (deduped to auction-solver grain).

In [ ]:
def revert_by_chain(df):
    rps = reward_per_auction_solver(df); rows = []
    for ch, d in df.groupby('blockchain'):
        p, lo, hi = wilson(int(d['reverted'].sum()), len(d))
        r = rps[rps['blockchain'] == ch]
        rows.append({'chain': ch, 'n_attempts': len(d), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'med_reward_usd': r['reward_usd'].median(),
            'med_penalty_usd': r['penalty_usd'].median(),
            'med_penalty_on_revert_usd': r.loc[r['reverted'], 'penalty_usd'].median()})
    return pd.DataFrame(rows).set_index('chain')

def plot_revert_by_chain(tbl):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    x = np.arange(len(tbl))

    lo = (tbl['revert_rate'] - tbl['ci_lo']).clip(lower=0).fillna(0)
    hi = (tbl['ci_hi'] - tbl['revert_rate']).clip(lower=0).fillna(0)

    ax[0].bar(
        x,
        tbl['revert_rate'] * 100,
        yerr=np.vstack([lo, hi]) * 100,
        capsize=4,
        color='#c0504d'
    )
    ax[0].set_xticks(x)
    ax[0].set_xticklabels([str(i) for i in tbl.index])
    ax[0].set_ylabel('revert rate (%)')
    ax[0].set_title('Revert rate by chain (95% Wilson CI)')

    w = 0.38
    ax[1].bar(
        x - w/2,
        tbl['med_reward_usd'],
        w,
        label='median reward',
        color='#4f81bd'
    )
    ax[1].bar(
        x + w/2,
        tbl['med_penalty_on_revert_usd'],
        w,
        label='median penalty on revert',
        color='#c0504d'
    )
    ax[1].set_xticks(x)
    ax[1].set_xticklabels([str(i) for i in tbl.index])
    ax[1].set_ylabel('USD, auction-solver grain')
    ax[1].set_title('Median reward vs median penalty on revert')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

T2 = revert_by_chain(DF); display(T2); plot_revert_by_chain(T2)

## 2. Revert rate by order size
Primary: per-chain quantile buckets (scale-adaptive), with median/min/max USD per bucket so Q-labels stay economically interpretable. Secondary: fixed-$ buckets for cross-chain comparability. Visual: boundary-free rolling revert rate over log10(size).

In [ ]:
def revert_by_quantile(df):
    d = add_chain_quantile_buckets(df); rows = []
    for (ch, q), g in d.groupby(['blockchain','size_q'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'size_q': q, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'med_size_usd': g['size_usd'].median(), 'min_size_usd': g['size_usd'].min(), 'max_size_usd': g['size_usd'].max(),
            'med_penalty_bps': g.loc[g['reverted'], 'penalty_bps'].median(),
            'med_gross_pnl_usd': g['gross_execution_pnl_usd'].median()})
    return pd.DataFrame(rows)

def revert_by_fixed(df):
    d = add_fixed_buckets(df); rows = []
    for (ch, b), g in d.groupby(['blockchain','size_fixed'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'bucket': b, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi})
    return pd.DataFrame(rows)

def plot_revert_by_size(qtbl, df):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for ch, g in qtbl.groupby('chain'):
        g = g.set_index('size_q').reindex([f'Q{i}' for i in range(1, 6)])
        lo = (g['revert_rate']-g['ci_lo']).clip(lower=0).fillna(0); hi = (g['ci_hi']-g['revert_rate']).clip(lower=0).fillna(0)
        ax[0].errorbar(range(len(g)), g['revert_rate']*100, yerr=np.vstack([lo, hi])*100, marker='o', capsize=3, label=ch)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels([f'Q{i}' for i in range(1, 6)])
    ax[0].set_xlabel('per-chain size quintile'); ax[0].set_ylabel('revert rate (%)')
    ax[0].set_title('Revert rate by per-chain size quantile'); ax[0].legend()
    for ch, d in df.groupby('blockchain'):
        v = d[['size_usd','reverted']].dropna(); v = v[v['size_usd'] > 0]
        if len(v) < MIN_PTS_LOGBIN: continue
        lx = np.log10(v['size_usd']); bins = np.linspace(lx.min(), lx.max(), 25); idx = np.digitize(lx, bins)
        xs, ys = [], []
        for b in range(1, len(bins)):
            m = idx == b
            if m.sum() >= MIN_PTS_PER_BIN: xs.append(lx[m].median()); ys.append(v['reverted'][m].mean()*100)
        ax[1].plot(xs, ys, marker='.', label=ch)
    ax[1].set_xlabel('log10(size_usd)'); ax[1].set_ylabel('revert rate (%)')
    ax[1].set_title('Log-binned revert rate vs size (boundary-free)'); ax[1].legend()
    plt.tight_layout(); plt.show()
    
def plot_penalty_bps_by_fixed_size(df):
    fixed_order = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']

    d = add_fixed_buckets(df.copy())
    d = d[d['reverted'] & d['penalty_bps'].notna()]

    tbl = (
        d.groupby(['size_fixed', 'blockchain'], observed=True)['penalty_bps']
        .median()
        .unstack('blockchain')
        .reindex(fixed_order)
    )

    display(tbl.round(2))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for ch in tbl.columns:
        ax.plot(tbl.index.astype(str), tbl[ch], marker='o', label=ch)

    ax.set_ylabel('median penalty on reverts (bps of size)')
    ax.set_xlabel('order-size bucket')
    ax.set_title('Fixed cap severity by order size')
    ax.tick_params(axis='x', rotation=25)
    ax.legend()
    plt.tight_layout()
    plt.show()

T4q = revert_by_quantile(DF); print('=== per-chain quantile (primary) ==='); display(T4q)
T4f = revert_by_fixed(DF); print('=== fixed-$ buckets (secondary) ===')
fixed_order = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']
display(T4f.pivot_table(index='bucket', columns='chain', values='revert_rate', observed=True).reindex(fixed_order))
plot_revert_by_size(T4q, DF)
plot_penalty_bps_by_fixed_size(DF)

## 3. Cap vs economics (reward / both P&L concepts)
Per-chain median reward, gross P&L (execution-only) and net P&L (incl reward/penalty) vs penalty cap. Answers if a higher penalty cap correlate with lower rewards and lower solver PnL?

In [ ]:
rps = reward_per_auction_solver(DF)

econ = rps.groupby('blockchain').agg(
    med_reward_usd=('reward_usd', 'median'),
    med_penalty_usd_all=('penalty_usd', 'median'),
    med_net_pnl_usd=('net_mechanism_pnl_usd', 'median'),
    revert_rate_solver=('reverted', 'mean'),
    share_penalized=('penalty_usd', lambda s: (s > 0).mean()),
)

penalty_on_revert = (
    rps[rps['reverted']]
    .groupby('blockchain')['penalty_usd']
    .median()
    .rename('med_penalty_on_revert_usd')
)

caps = DF.groupby('blockchain').agg(
    penalty_cap_tok=('penalty_cap_tok', 'median'),
    penalty_cap_tok_unique=('penalty_cap_tok', 'nunique'),
    median_penalty_cap_usd=('penalty_cap_usd', 'median'),

    reward_cap_tok=('reward_cap_upper_tok', 'median'),
    reward_cap_tok_unique=('reward_cap_upper_tok', 'nunique'),
    median_reward_cap_usd=('reward_cap_upper_usd', 'median'),

    med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
    revert_rate_attempt=('reverted', 'mean'),
    median_size_usd=('size_usd', 'median'),
    n_attempts=('order_uid', 'size'),
    n_orders=('order_uid', 'nunique'),
)

T5 = (
    caps
    .join(econ)
    .join(penalty_on_revert)
    .sort_values('median_penalty_cap_usd')
)
T5['reward_cap_minus_penalty_cap_usd'] = (
    T5['median_reward_cap_usd'] - T5['median_penalty_cap_usd']
)
T5['reward_cap_to_penalty_cap'] = (
    T5['median_reward_cap_usd'] / T5['median_penalty_cap_usd']
)
print('=== caps vs solver economics ===')
display(T5.round({
    'penalty_cap_tok': 4,
    'median_penalty_cap_usd': 4,
    'reward_cap_tok': 4,
    'median_reward_cap_usd': 4,
    'med_reward_usd': 4,
    'med_penalty_usd_all': 4,
    'med_penalty_on_revert_usd': 4,
    'med_gross_pnl_usd': 4,
    'med_net_pnl_usd': 4,
    'revert_rate_attempt': 4,
    'revert_rate_solver': 4,
    'share_penalized': 4,
    'median_size_usd': 2,
    'reward_cap_minus_penalty_cap_usd': 4,
    'reward_cap_to_penalty_cap': 4,
}))
if len(T5) == 1:
    print(
        f"Only one chain loaded ({T5.index[0]}): this section is a summary only. "
        "Load the second chain to compare cap direction."
    )

else:
    low_chain = T5.index[0]
    high_chain = T5.index[-1]

    direction = pd.DataFrame({
        'low_penalty_cap_chain': low_chain,
        'high_penalty_cap_chain': high_chain,
        'low_penalty_cap_value': T5.iloc[0],
        'high_penalty_cap_value': T5.iloc[-1],
        'high_minus_low': T5.iloc[-1] - T5.iloc[0],
    })[
        [
            'low_penalty_cap_chain',
            'high_penalty_cap_chain',
            'low_penalty_cap_value',
            'high_penalty_cap_value',
            'high_minus_low',
        ]
    ]

    direction = direction.loc[
        [
            'median_penalty_cap_usd',
            'median_reward_cap_usd',
            'med_reward_usd',
            'med_gross_pnl_usd',
            'med_net_pnl_usd',
            'revert_rate_attempt',
            'revert_rate_solver',
            'share_penalized',
            'med_penalty_on_revert_usd',
            'median_size_usd',
        ]
    ]

    if len(T5) > 2:
        print(
            f"{len(T5)} chains loaded: table sorted by median penalty cap USD; "
            "direction below compares the lowest-cap vs highest-cap chain."
        )
    print('=== direction from lower penalty cap chain to higher penalty cap chain ===')
    display(direction.round(4))

if len(T5) >= 2:
    plot_df = T5.sort_values('median_penalty_cap_usd')

    y_metrics = [
        ('med_reward_usd', 'Median reward'),
        ('med_gross_pnl_usd', 'Median gross PnL'),
        ('med_net_pnl_usd', 'Median net PnL'),
    ]

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))

    for a, (ycol, ylabel) in zip(ax, y_metrics):
        a.plot(
            plot_df['median_penalty_cap_usd'],
            plot_df[ycol],
            marker='o',
            ls='none',
        )

        for ch, r in plot_df.iterrows():
            label = (
                f"{ch}\n"
                f"reward cap=${r['median_reward_cap_usd']:.2f}"
            )
            a.annotate(
                label,
                (r['median_penalty_cap_usd'], r[ycol]),
                textcoords='offset points',
                xytext=(6, 6),
                fontsize=9,
            )

        a.axhline(0, linewidth=1, alpha=0.4)
        a.set_xlabel('Median penalty cap, USD')
        a.set_ylabel(f'{ylabel}, USD')
        a.set_title(f'{ylabel} vs penalty cap')

    plt.tight_layout()
    plt.show()

## 4. Cap vs markout (surplus delivered to users)
Markout is measured in bps on settled executions; test whether higher-penalty-cap chains show worse median markout, with ECDFs comparing the full clipped distribution.

In [ ]:
def markout_vs_cap(df):
    d = add_fixed_buckets(df[df['settled'] == True])
    by_chain = d.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'), reward_cap_usd=('reward_cap_upper_usd', 'median'),
        med_markout_bps=('markout_bps','median'), n=('order_uid','size'), n_markout=('markout_bps', 'count'),
        median_size_usd=('size_usd', 'median')).sort_values('penalty_cap_usd')
    by_size = d.groupby(['blockchain','size_fixed'], observed=True)['markout_bps'].median().unstack(0)
    return by_chain, by_size, d

def plot_markout(by_chain, d):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    if len(by_chain) >= 2:
        p = by_chain.sort_values('penalty_cap_usd')

        ax[0].plot(
            p['penalty_cap_usd'],
            p['med_markout_bps'],
            marker='o',
            ls='none',
            color='#9bbb59'
        )

        for ch, r in p.iterrows():
            ax[0].annotate(
                f"{ch}\nreward cap=${r['reward_cap_usd']:.2f}",
                (r['penalty_cap_usd'], r['med_markout_bps']),
                textcoords='offset points',
                xytext=(6, 6),
                fontsize=9,
            )

        ax[0].set_xlabel('median penalty cap (USD)')
        ax[0].set_ylabel('median markout (bps)')
        ax[0].set_title('Median markout vs penalty cap')
    else:
        barpos(ax[0], list(by_chain.index), by_chain['med_markout_bps'], color='#9bbb59')
        ax[0].set_xlabel('chain')
        ax[0].set_ylabel('median markout (bps)')
        ax[0].set_title('Surplus to users by chain')

    ax[0].axhline(0, color='k', lw=.8)

    for ch, g in d.groupby('blockchain'):
        ecdf(ax[1], g['markout_bps'].clip(-200, 200), ch)

    ax[1].set_xlabel('markout (bps, clipped ±200)')
    ax[1].set_ylabel('ECDF')
    ax[1].set_title('Markout distribution')
    ax[1].legend()

    plt.tight_layout()
    plt.show()

T6c, T6s, T6d = markout_vs_cap(DF); display(T6c); display(T6s); plot_markout(T6c, T6d)

## 5. Three outcomes vs cap
Compare solver gross execution P&L, settled-user markout, and time-to-happy-moo against penalty cap across chains. Hypothesis: higher caps are associated with worse markout and longer time-to-happy-moo.
First: chain-level view.
Second: same analysis restricted to top traded token pairs per chain, where sell/buy and buy/sell are treated as the same pair.

In [ ]:
TOP_N_PAIRS_PER_CHAIN = 3
MIN_SETTLED_FOR_PAIR_RANKING = 30

BLOCK_SECONDS = {
    'polygon': 1.75,
    'bnb': 0.5,
    'ethereum': 12.0,
    'gnosis': 5.0,
    'base': 2.0,
    'avalanche_c': 1.0,
    'arbitrum': 0.25
}
def _short_addr(x, n=6):
    if pd.isna(x):
        return 'NA'
    s = str(x)
    return s[:n + 2] if s.startswith('0x') else s[:n]
def _token_label(df, token_col):
    symbol_candidates = [
        f'{token_col}_symbol',
        f'{token_col}_token_symbol',
        f'{token_col}_name',
    ]

    for c in symbol_candidates:
        if c in df.columns:
            sym = df[c].replace(NIL).astype('object')
            addr = df[token_col].astype(str).map(_short_addr)
            return sym.where(sym.notna(), addr).astype(str)

    return df[token_col].astype(str).map(_short_addr)


def add_unordered_pair_key(df):
    d = df.copy()

    d['_sell_addr'] = d['sell_token'].astype(str).str.lower().str.strip()
    d['_buy_addr'] = d['buy_token'].astype(str).str.lower().str.strip()

    d['_sell_label'] = _token_label(d, 'sell_token')
    d['_buy_label'] = _token_label(d, 'buy_token')

    sell_first = d['_sell_addr'] <= d['_buy_addr']

    d['pair_token_0'] = np.where(sell_first, d['_sell_addr'], d['_buy_addr'])
    d['pair_token_1'] = np.where(sell_first, d['_buy_addr'], d['_sell_addr'])

    d['pair_label_0'] = np.where(sell_first, d['_sell_label'], d['_buy_label'])
    d['pair_label_1'] = np.where(sell_first, d['_buy_label'], d['_sell_label'])

    d['pair_key'] = d['pair_token_0'] + ' / ' + d['pair_token_1']
    d['pair_label'] = d['pair_label_0'].astype(str) + ' / ' + d['pair_label_1'].astype(str)

    return d


def top_pairs_by_chain_volume(df, top_n=TOP_N_PAIRS_PER_CHAIN):
    d = add_unordered_pair_key(df)

    settled = d[d['settled'] == True].copy()

    pair_volume = (
        settled
        .groupby(['blockchain', 'pair_key'], as_index=False)
        .agg(
            pair_label=('pair_label', 'first'),
            pair_volume_usd=('volume_usd', 'sum'),
            settled_attempts=('order_uid', 'size'),
            settled_orders=('order_uid', 'nunique'),
            n_markout=('markout_bps', lambda s: s.notna().sum()),
            n_tthm=('seconds_to_settle', lambda s: s.notna().sum()),
        )
    )

    pair_volume = pair_volume[
        (pair_volume['settled_attempts'] >= MIN_SETTLED_FOR_PAIR_RANKING)
        & (pair_volume['n_markout'] >= MIN_SETTLED_FOR_PAIR_RANKING)
        & (pair_volume['n_tthm'] >= MIN_SETTLED_FOR_PAIR_RANKING)
    ].copy()

    pair_volume = pair_volume.sort_values(
        ['blockchain', 'pair_volume_usd'],
        ascending=[True, False],
    )

    top_pairs = (
        pair_volume
        .groupby('blockchain', group_keys=False)
        .head(top_n)
        .copy()
    )

    top_pairs['pair_rank'] = (
        top_pairs
        .sort_values(['blockchain', 'pair_volume_usd'], ascending=[True, False])
        .groupby('blockchain')
        .cumcount()
        .add(1)
    )

    return top_pairs, d


def three_outcomes(df):
    st = df[df['settled'] == True]

    base = df.groupby('blockchain').agg(
        penalty_cap_usd=('penalty_cap_usd', 'median'),
        reward_cap_usd=('reward_cap_upper_usd', 'median'),
        med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
    )

    return base.join([
        st.groupby('blockchain')['markout_bps'].median().rename('med_markout_bps'),
        st.groupby('blockchain')['seconds_to_settle'].median().rename('med_tthm_s'),
        st.groupby('blockchain')['markout_bps'].count().rename('n_markout'),
        st.groupby('blockchain')['seconds_to_settle'].count().rename('n_tthm'),
    ]).sort_values('penalty_cap_usd')


def three_outcomes_top_pairs(df, top_n=TOP_N_PAIRS_PER_CHAIN):
    top_pairs, d = top_pairs_by_chain_volume(df, top_n=top_n)

    d = d.merge(
        top_pairs[['blockchain', 'pair_key', 'pair_rank']],
        on=['blockchain', 'pair_key'],
        how='inner',
    )

    st = d[d['settled'] == True].copy()

    base = (
        d.groupby(['blockchain', 'pair_key', 'pair_label', 'pair_rank'], as_index=False)
        .agg(
            penalty_cap_usd=('penalty_cap_usd', 'median'),
            reward_cap_usd=('reward_cap_upper_usd', 'median'),
            med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
            attempts=('order_uid', 'size'),
            orders=('order_uid', 'nunique'),
        )
    )

    settled_outcomes = (
        st.groupby(['blockchain', 'pair_key', 'pair_label', 'pair_rank'], as_index=False)
        .agg(
            med_markout_bps=('markout_bps', 'median'),
            med_tthm_s=('seconds_to_settle', 'median'),
            n_markout=('markout_bps', lambda s: s.notna().sum()),
            n_tthm=('seconds_to_settle', lambda s: s.notna().sum()),
            n_settled=('order_uid', 'size'),
            settled_volume_usd=('volume_usd', 'sum'),
            median_size_usd=('size_usd', 'median'),
        )
    )

    out = base.merge(
        settled_outcomes,
        on=['blockchain', 'pair_key', 'pair_label', 'pair_rank'],
        how='left',
    )

    out = out.merge(
        top_pairs[
            [
                'blockchain',
                'pair_key',
                'pair_volume_usd',
                'settled_attempts',
                'settled_orders',
            ]
        ],
        on=['blockchain', 'pair_key'],
        how='left',
    )

    out['med_tthm_blocks'] = (
        out['med_tthm_s'] / out['blockchain'].map(BLOCK_SECONDS)
    )

    out['display_label'] = (
        out['blockchain']
        + ' #'
        + out['pair_rank'].astype(int).astype(str)
        + ' / '
        + out['pair_label']
    )

    return out.sort_values(['blockchain', 'pair_rank'])


def collapse_top_pairs_to_chain_medians(pair_tbl):
    chain_tbl = (
        pair_tbl
        .groupby('blockchain', as_index=False)
        .agg(
            penalty_cap_usd=('penalty_cap_usd', 'median'),
            reward_cap_usd=('reward_cap_usd', 'median'),
            med_gross_pnl_usd=('med_gross_pnl_usd', 'median'),
            med_markout_bps=('med_markout_bps', 'median'),
            med_tthm_s=('med_tthm_s', 'median'),
            med_tthm_blocks=('med_tthm_blocks', 'median'),
            top_pair_volume_usd=('pair_volume_usd', 'sum'),
            top_pairs_used=('pair_key', 'nunique'),
            total_settled_attempts=('n_settled', 'sum'),
            total_markout_obs=('n_markout', 'sum'),
            total_tthm_obs=('n_tthm', 'sum'),
            median_size_usd=('median_size_usd', 'median'),
        )
    )

    pair_list = (
        pair_tbl
        .sort_values(['blockchain', 'pair_rank'])
        .groupby('blockchain')['pair_label']
        .apply(lambda s: ' | '.join(s.astype(str)))
        .rename('top_pair_labels')
        .reset_index()
    )

    chain_tbl = chain_tbl.merge(pair_list, on='blockchain', how='left')

    return chain_tbl.sort_values('penalty_cap_usd')


def _corr_stats(tbl, xcol, ycol):
    clean = (
        tbl[[xcol, ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(clean) < 3 or clean[xcol].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan

    spearman_rho, spearman_p = stats.spearmanr(clean[xcol], clean[ycol])
    pearson_r, pearson_p = stats.pearsonr(clean[xcol], clean[ycol])

    return spearman_rho, spearman_p, pearson_r, pearson_p


def plot_three(tbl):
    metrics = [
        ('med_gross_pnl_usd', 'solver gross P&L (USD)'),
        ('med_markout_bps', 'markout (bps, user)'),
        ('med_tthm_s', 'time to execution (s)'),
        ('med_tthm_blocks', 'time to execution (blocks)'),
    ]

    fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))

    for (m, lab), a in zip(metrics, ax):
        if len(tbl) >= 2:
            p = tbl.sort_values('penalty_cap_usd')

            a.plot(
                p['penalty_cap_usd'],
                p[m],
                marker='o',
                ls='none',
            )

            for ch, r in p.iterrows():
                a.annotate(
                    f"{ch}\nreward cap=${r['reward_cap_usd']:.2f}",
                    (r['penalty_cap_usd'], r[m]),
                    textcoords='offset points',
                    xytext=(6, 6),
                    fontsize=9,
                )

            a.set_xlabel('penalty cap (USD)')
        else:
            barpos(a, list(tbl.index), tbl[m])
            a.set_xlabel('chain')

        a.axhline(0, color='k', lw=.8, alpha=.4)
        a.set_ylabel(lab)
        a.set_title(lab)

    plt.tight_layout()
    plt.show()


def plot_top_pair_chain_medians(tbl):
    outcomes = [
        ('med_markout_bps', 'Median markout', 'bps'),
        ('med_tthm_s', 'Median time to execution', 'seconds'),
    ]

    if len(tbl) < 3:
        print(f'Need at least 3 chain observations; have {len(tbl)}.')
        return

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            'Penalty cap vs markout',
            'Penalty cap vs time-to-happy-moo',
        ],
        horizontal_spacing=0.10,
    )

    max_volume = tbl['top_pair_volume_usd'].max()

    for col_idx, (ycol, ylabel, unit) in enumerate(outcomes, start=1):
        plot_tbl = (
            tbl.replace([np.inf, -np.inf], np.nan)
            .dropna(subset=['penalty_cap_usd', ycol])
            .copy()
        )

        if plot_tbl.empty:
            continue

        marker_size = 14 + 36 * (plot_tbl['top_pair_volume_usd'] / max_volume)

        fig.add_trace(
            go.Scatter(
                x=plot_tbl['penalty_cap_usd'],
                y=plot_tbl[ycol],
                mode='markers+text',
                text=plot_tbl['blockchain'],
                textposition='top center',
                name=ylabel,
                showlegend=False,
                marker=dict(
                    size=marker_size,
                    opacity=0.8,
                    line=dict(width=0.9, color='black'),
                ),
                customdata=np.stack(
                    [
                        plot_tbl['blockchain'],
                        plot_tbl['top_pairs_used'],
                        plot_tbl['top_pair_labels'],
                        plot_tbl['top_pair_volume_usd'],
                        plot_tbl['total_settled_attempts'],
                        plot_tbl['total_markout_obs'],
                        plot_tbl['total_tthm_obs'],
                        plot_tbl['reward_cap_usd'],
                        plot_tbl['median_size_usd'],
                    ],
                    axis=-1,
                ),
                hovertemplate=(
                    '<b>%{customdata[0]}</b><br>'
                    'Penalty cap: $%{x:,.2f}<br>'
                    f'{ylabel}: %{{y:,.2f}} {unit}<br>'
                    'Top pairs used: %{customdata[1]:.0f}<br>'
                    'Pairs: %{customdata[2]}<br>'
                    'Top-pair settled volume: $%{customdata[3]:,.0f}<br>'
                    'Settled attempts: %{customdata[4]:,.0f}<br>'
                    'Markout obs: %{customdata[5]:,.0f}<br>'
                    'TTHM obs: %{customdata[6]:,.0f}<br>'
                    'Reward cap: $%{customdata[7]:,.4f}<br>'
                    'Median size: $%{customdata[8]:,.2f}'
                    '<extra></extra>'
                ),
            ),
            row=1,
            col=col_idx,
        )

        spearman_rho, spearman_p, pearson_r, pearson_p = _corr_stats(
            plot_tbl,
            'penalty_cap_usd',
            ycol,
        )

        fig.add_annotation(
            xref=f'x{col_idx} domain' if col_idx > 1 else 'x domain',
            yref=f'y{col_idx} domain' if col_idx > 1 else 'y domain',
            x=0.02,
            y=0.98,
            showarrow=False,
            align='left',
            bgcolor='rgba(255,255,255,0.75)',
            bordercolor='rgba(0,0,0,0.25)',
            borderwidth=1,
            text=(
                f'Spearman ρ={spearman_rho:.2f}, p={spearman_p:.3f}<br>'
                f'Pearson r={pearson_r:.2f}, p={pearson_p:.3f}<br>'
                f'n={len(plot_tbl)} chains'
            ),
            row=1,
            col=col_idx,
        )

        fig.update_xaxes(
            title_text='Penalty cap, USD',
            type='linear',
            tickprefix='$',
            tickformat=',.0f',
            row=1,
            col=col_idx,
        )

        fig.update_yaxes(
            title_text=f'{ylabel} ({unit})',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='black',
            row=1,
            col=col_idx,
        )

    fig.update_layout(
        title=(
            f'Penalty cap vs outcomes using median of top '
            f'{TOP_N_PAIRS_PER_CHAIN} settled-volume token pairs per chain'
        ),
        width=1500,
        height=650,
        template='plotly_white',
    )

    fig.show()

T7 = three_outcomes(DF)

T7['med_tthm_blocks'] = (
    T7['med_tthm_s'] / T7.index.to_series().map(BLOCK_SECONDS)
)

display(T7.round({
    'penalty_cap_usd': 4,
    'reward_cap_usd': 4,
    'med_gross_pnl_usd': 4,
    'med_markout_bps': 2,
    'med_tthm_s': 2,
    'med_tthm_blocks': 2,
}))

plot_three(T7)

T7_pairs = three_outcomes_top_pairs(
    DF,
    top_n=TOP_N_PAIRS_PER_CHAIN,
)

print(f"=== top {TOP_N_PAIRS_PER_CHAIN} settled-volume token pairs per chain ===")
display(
    T7_pairs[
        [
            'blockchain',
            'pair_rank',
            'pair_label',
            'pair_volume_usd',
            'settled_attempts',
            'settled_orders',
            'n_settled',
            'n_markout',
            'n_tthm',
            'penalty_cap_usd',
            'reward_cap_usd',
            'med_gross_pnl_usd',
            'med_markout_bps',
            'med_tthm_s',
            'med_tthm_blocks',
            'median_size_usd',
        ]
    ]
    .sort_values(['blockchain', 'pair_rank'])
    .round({
        'pair_volume_usd': 2,
        'penalty_cap_usd': 4,
        'reward_cap_usd': 4,
        'med_gross_pnl_usd': 4,
        'med_markout_bps': 2,
        'med_tthm_s': 2,
        'med_tthm_blocks': 2,
        'median_size_usd': 2,
    })
)

print("pair count by chain:")
display(T7_pairs.groupby('blockchain')['pair_key'].nunique())

T7_pair_chain_medians = collapse_top_pairs_to_chain_medians(T7_pairs)

print(f"=== chain medians across top {TOP_N_PAIRS_PER_CHAIN} settled-volume pairs ===")
display(
    T7_pair_chain_medians[
        [
            'blockchain',
            'top_pairs_used',
            'top_pair_volume_usd',
            'total_settled_attempts',
            'penalty_cap_usd',
            'reward_cap_usd',
            'med_gross_pnl_usd',
            'med_markout_bps',
            'med_tthm_s',
            'med_tthm_blocks',
            'median_size_usd',
            'top_pair_labels',
        ]
    ]
    .round({
        'top_pair_volume_usd': 2,
        'penalty_cap_usd': 4,
        'reward_cap_usd': 4,
        'med_gross_pnl_usd': 4,
        'med_markout_bps': 2,
        'med_tthm_s': 2,
        'med_tthm_blocks': 2,
        'median_size_usd': 2,
    })
)

corr_rows = []

for ycol, label in [
    ('med_markout_bps', 'median markout bps across top pairs'),
    ('med_tthm_s', 'median time to execution seconds across top pairs'),
    ('med_tthm_blocks', 'median time to execution blocks across top pairs'),
]:
    clean = (
        T7_pair_chain_medians[['penalty_cap_usd', ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(clean) >= 3 and clean['penalty_cap_usd'].nunique() >= 2:
        spearman_rho, spearman_p = stats.spearmanr(
            clean['penalty_cap_usd'],
            clean[ycol],
        )

        pearson_r, pearson_p = stats.pearsonr(
            clean['penalty_cap_usd'],
            clean[ycol],
        )
    else:
        spearman_rho = spearman_p = pearson_r = pearson_p = np.nan

    corr_rows.append({
        'outcome': label,
        'n_chains': len(clean),
        'spearman_rho': spearman_rho,
        'spearman_p': spearman_p,
        'pearson_r': pearson_r,
        'pearson_p': pearson_p,
    })

print("=== top-pair chain-median correlation summary ===")
display(pd.DataFrame(corr_rows).round(4))

plot_top_pair_chain_medians(T7_pair_chain_medians)

## 6. Distribution of capped vs uncapped penalties, per chain.
On reverts: how often the cap binds and how much penalty it forgives. 

In [ ]:
def cap_binding(df):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted']]; rows = []
    for ch, d in rev.groupby('blockchain'):
        unc = d['penalty_uncapped_usd']; cap = d['penalty_usd']; w = unc.quantile(0.999)
        rows.append({'chain': ch, 'reverts': len(d), 'cap_binds_rate': d['cap_binds'].mean(),
            'med_penalty_on_revert_usd': cap.median(), 'capped_sum_usd': cap.sum(), 'uncapped_sum_usd': unc.sum(),
            'forgiven_sum_usd': (unc-cap).sum(), 'forgiven_sum_winsor_usd': (np.minimum(unc, w)-cap).sum()})
    return pd.DataFrame(rows).set_index('chain')

def cap_percentile(df):
    rps = reward_per_auction_solver(df)
    rev = rps[rps['reverted'] & (rps['penalty_uncapped_usd'] > 0) & (rps['penalty_cap_usd'] > 0)]
    rows = []
    for ch, d in rev.groupby('blockchain'):
        unc = d['penalty_uncapped_usd'].values
        cap = d['penalty_cap_usd'].median()
        rows.append(dict(chain=ch, cap_usd=round(cap, 2),
                         cap_percentile=round((unc <= cap).mean() * 100, 1),
                         cap_binds_rate=round((unc > cap).mean() * 100, 1),
                         unc_p50=round(np.median(unc), 2),
                         unc_p90=round(np.quantile(unc, .9), 2),
                         unc_p99=round(np.quantile(unc, .99), 2)))
    return pd.DataFrame(rows).set_index('chain')

def plot_cap_binding(df, tbl):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted'] & (rev['penalty_uncapped_usd'] > 0)]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    chains = list(rev['blockchain'].unique())
    cmap = {c: col for c, col in zip(chains, plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1))))}
    for ch, d in rev.groupby('blockchain'):
        ecdf(ax[0], np.log10(d['penalty_uncapped_usd'].clip(lower=1e-6)), f'{ch} uncapped', color=cmap[ch])
        ax[0].axvline(np.log10(d['penalty_cap_usd'].median()), ls='--', alpha=.6, color=cmap[ch])
    ax[0].set_xlabel('log10 uncapped penalty (USD)'); ax[0].set_ylabel('ECDF')
    ax[0].set_title('Uncapped penalty vs cap (dashed)'); ax[0].legend(fontsize=8)
    barpos(ax[1], list(tbl.index), tbl['cap_binds_rate']*100, color='#c0504d')
    ax[1].set_ylabel('% reverts where cap binds'); ax[1].set_title('Cap-binding rate')
    plt.tight_layout(); plt.show()
def plot_uncapped_over_cap(df):
    rps = reward_per_auction_solver(df).copy()
    rev = rps[
        rps['reverted']
        & (rps['penalty_uncapped_usd'] > 0)
        & (rps['penalty_cap_usd'] > 0)
    ].copy()
    rev['uncapped_over_cap'] = (
        rev['penalty_uncapped_usd'] / rev['penalty_cap_usd']
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    for ch, d in rev.groupby('blockchain'):
        ecdf(ax, d['uncapped_over_cap'].clip(lower=1e-6), ch)
    ax.axvline(1, color='k', ls='--', lw=1, alpha=.7)
    ax.set_xscale('log')
    ax.set_xlabel('uncapped penalty / cap')
    ax.set_ylabel('ECDF')
    ax.set_title('Uncapped penalty relative to cap')
    ax.legend()

    plt.tight_layout()
    plt.show()
T8 = cap_binding(DF); display(T8); plot_cap_binding(DF, T8); plot_uncapped_over_cap(DF)
T8b = cap_percentile(DF)
print("Cap percentile placement:")
display(T8b)

## 7. Variable-rate penalty counterfactual
Recompute penalties had the cap been a variable rate (% of size): `min(uncapped, r x volume)`: a variable cap, bidding held constant. Report collected penalty across a grid of rates, the revenue-neutral rate per chain (exact, capped bisection), and who pays more/less by size band and solver.

In [ ]:
def variable_rate(df, grid=(1, 2, 5, 10, 20, 50)):
    rev = reward_per_auction_solver(df)
    rev = rev[rev['reverted'] & rev['size_usd'].notna() & (rev['size_usd'] > 0)].copy()
    rev = rev[rev.groupby('blockchain')['size_usd'].transform(lambda s: s < s.quantile(0.999))]  
    def collected(unc, size, r):                       
        return np.minimum(unc, r / 1e4 * size).sum()
    def neutral_bps(unc, size, target):                # exact rev-neutral (sum is monotone in r)
        lo, hi = 0.0, 1000.0
        for _ in range(60):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if collected(unc, size, mid) < target else (lo, mid)
        return (lo + hi) / 2
    sweep_rows, neutral, parts = [], {}, []
    for ch, d in rev.groupby('blockchain'):
        unc, size, current = d.penalty_uncapped_usd.values, d.size_usd.values, d.penalty_usd.sum()
        for r in grid:
            sweep_rows.append(dict(chain=ch, rate_bps=r, total_usd=collected(unc, size, r),
                                   ratio=collected(unc, size, r) / current if current else np.nan))
        neutral[ch] = neutral_bps(unc, size, current)
        parts.append(d.assign(pen_fixed=d.penalty_usd,
                              pen_var=np.minimum(d.penalty_uncapped_usd, neutral[ch] / 1e4 * d.size_usd)))
    gw = pd.concat(parts)
    bands  = [0, 100, 1_000, 10_000, 100_000, np.inf]
    labels = ["<$100", "$100-1k", "$1k-10k", "$10k-100k", ">$100k"]
    gw["band"] = pd.cut(gw.size_usd, bands, labels=labels)
    return pd.DataFrame(sweep_rows), neutral, gw, labels

def share_shift(gw, key, chain=None):
    d = gw if chain is None else gw[gw.blockchain == chain]
    tf, tv = d.pen_fixed.sum(), d.pen_var.sum()
    s = d.groupby(key, observed=True).agg(fixed=("pen_fixed", "sum"), variable=("pen_var", "sum"))
    s["fixed_share"], s["var_share"] = s.fixed / tf, s.variable / tv
    s["delta_pp"] = (s.var_share - s.fixed_share) * 100
    return s

def plot_variable(sweep, neutral, gw, labels):
    chains = list(neutral)
    colors = {ch: c for ch, c in zip(chains, plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1))))}
    fig, ax = plt.subplots(2, 2, figsize=(14, 9))
    for ch in chains:
        d = sweep[sweep.chain == ch]
        ax[0,0].plot(d.rate_bps, d.ratio, marker='o', color=colors[ch],
                     label=f"{ch} (neutral ≈ {neutral[ch]:.1f} bps)")
        ax[0,0].axvline(neutral[ch], color=colors[ch], ls=':', lw=1)
    ax[0,0].axhline(1.0, color='gray', ls='--', lw=.8)
    ax[0,0].set(xlabel="rate (bps of size)", ylabel="collected / current",
                title="Variable rate vs current flat cap\n(1.0 = revenue neutral; dotted = neutral rate)")
    ax[0,0].legend()
    for ch in chains:
        d = gw[gw.blockchain == ch].copy()
        if len(d) < MIN_PTS_DECILE: continue
        d["dec"] = pd.qcut(d.size_usd, 10, labels=False, duplicates="drop")
        eff = d.groupby("dec").apply(lambda x: x.pen_fixed.sum() / x.size_usd.sum() * 1e4,
                                     include_groups=False)
        msz = d.groupby("dec").size_usd.median()
        ax[0,1].plot(msz.values, eff.values, 'o-', color=colors[ch], label=ch)
    for ch in chains:
        ax[0,1].axhline(neutral[ch], color=colors[ch], ls=':', lw=1)
    ax[0,1].set(xscale="log", yscale="log", xlabel="median order size in decile (USD)",
                ylabel="effective penalty (bps of size)",
                title="Current fixed cap is REGRESSIVE\n(flat rate = horizontal; dotted = neutral rate)")
    ax[0,1].legend()
    bd = pd.DataFrame({ch: share_shift(gw, "band", ch)["delta_pp"].reindex(labels)
                       for ch in chains})
    bd.plot.bar(ax=ax[1,0], color=[colors[ch] for ch in chains], width=.8)
    ax[1,0].axhline(0, color='k', lw=.8)
    ax[1,0].set(ylabel="budget-share shift (pp)", xlabel="order size band",
                title="Who pays more under the rate (variable − fixed)\n+ve = pays more")
    ax[1,0].tick_params(axis='x', rotation=30); ax[1,0].legend(title=None)
    gs = share_shift(gw, "solver_name").assign(n=gw.groupby("solver_name").size())
    gs = gs[gs.n >= 20].sort_values("delta_pp")
    pick = pd.concat([gs.head(5), gs.tail(5)])
    bar_c = ['#2f7d62' if v < 0 else '#b1442f' for v in pick.delta_pp]
    ax[1,1].barh(range(len(pick)), pick.delta_pp, color=bar_c)
    ax[1,1].set_yticks(range(len(pick))); ax[1,1].set_yticklabels(pick.index, fontsize=8)
    ax[1,1].axvline(0, color='k', lw=.8)
    ax[1,1].set(xlabel="budget-share shift (pp)",
                title="Solver winners (green) & losers (red)\nunder revenue-neutral rate")
    plt.tight_layout(); plt.show()
sweep, neutral, gw, labels = variable_rate(DF)
print("revenue-neutral rate (bps of size, capped):", {k: round(v, 2) for k, v in neutral.items()})
display(sweep.round(2))
plot_variable(sweep, neutral, gw, labels)

## 8. Sit-out (non-economic penalty) simulation
Suspend a reverting solver for N auctions (exponential backoff, `block_deadline` order): tally forgone reward, removed attempts, coverage-at-risk. The triggering revert is still penalised, future in-window penalties are a diagnostic only (not credited). Does not change overbidding incentives.

In [ ]:
def sit_out(df, base=1, factor=2, max_power=5, reset_clean=10):
    cap = factor ** max_power
    rps = reward_per_auction_solver(df)
    bd = (df.groupby(['blockchain', 'auction_id', 'solver'])['block_deadline']
            .first().reset_index())
    base_df = rps.merge(bd, on=['blockchain', 'auction_id', 'solver'], how='left')
    settled = base_df[~base_df['reverted']]
    osettle = ({ch: {o: list(zip(gg['block_deadline'], gg['solver']))
                     for o, gg in g.groupby('order_uid')}
                for ch, g in settled.groupby('blockchain')}
               if 'order_uid' in base_df else {})

    res = []
    for (ch, solver), d in base_df.sort_values('block_deadline').groupby(['blockchain', 'solver']):
        d = d.reset_index(drop=True)
        rev = d['reverted'].to_numpy()
        rew = d['reward_usd'].fillna(0).to_numpy(float)
        pen = d['penalty_usd'].fillna(0).to_numpy(float)
        oc  = d['order_count'].fillna(1).to_numpy() if 'order_count' in d else np.ones(len(d))
        oid = d['order_uid'].to_numpy() if 'order_uid' in d else np.array([None] * len(d))
        bdl = d['block_deadline'].to_numpy(float)

        ban = offense = clean = 0
        trig_pen = forg = future_pen_diag = 0.0
        removed = removed_settled = removed_rev = covg = 0

        for i in range(len(d)):
            if ban > 0:                                  # suspended: drop this attempt
                ban -= 1; removed += int(oc[i])
                if not rev[i]:
                    removed_settled += 1; forg += rew[i]
                    covered = any(o_bd > bdl[i] and o_sv != solver
                                  for o_bd, o_sv in osettle.get(ch, {}).get(oid[i], []))
                    if not covered: covg += int(oc[i])
                else:
                    removed_rev += 1; future_pen_diag += pen[i]   
                continue
            if rev[i]:                                   # active revert: trigger / escalate
                offense += 1; clean = 0
                ban = int(min(cap, base * factor ** (offense - 1)))
                trig_pen += pen[i]                       # triggering penalty still incurred
            else:                                        # active clean attempt: maybe reset
                clean += 1
                if clean >= reset_clean: offense = clean = 0

        res.append({
            'chain': ch,
            'solver': d['solver_name'].iloc[0] if d['solver_name'].notna().any() else solver,
            'reverts': int(rev.sum()),
            'triggering_penalty_usd': trig_pen,
            'forgone_reward_usd': forg,
            'removed_orders': removed,
            'removed_settled_wins': removed_settled,
            'removed_reverted_wins': removed_rev,
            'coverage_at_risk_orders_ub': covg,
            'future_penalty_diag_usd': future_pen_diag,  
        })

    out = pd.DataFrame(res)
    out['sitout_cost_usd'] = out['forgone_reward_usd']  
    return out.sort_values('sitout_cost_usd', ascending=False)

def plot_sit_out(tbl, top=15):
    t = tbl.head(top)
    labels = [f"{c} / {s}" for c, s in zip(t['chain'], t['solver'])][::-1]
    y = np.arange(len(labels)); w = 0.4
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(y + w/2, t['forgone_reward_usd'][::-1], w, label='forgone reward', color='#8064a2')
    ax.barh(y - w/2, t['coverage_at_risk_orders_ub'][::-1], w, label='coverage-at-risk (orders, ub)', color='#4f81bd')
    ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.axvline(0, color='k', lw=.8)
    ax.set_xlabel('per-solver sit-out cost')
    ax.set_title(f'Sit-out opportunity cost by chain/solver (top {top})')
    ax.legend(); plt.tight_layout(); plt.show()
    
SCHEDULES = {
    "1 auction (flat)":   dict(base=1, factor=1, max_power=0),   
    "1,2,4,... cap 32":   dict(base=1, factor=2, max_power=5),   
    "1,3,9,... cap 81":   dict(base=1, factor=3, max_power=4),   
    "1,2,4,... cap 8":    dict(base=1, factor=2, max_power=3),   
}

rows = []
detail = {}
for name, kw in SCHEDULES.items():
    t = sit_out(DF, **kw); detail[name] = t
    g = t.groupby('chain').agg(
        solvers=('solver', 'nunique'),
        reverts=('reverts', 'sum'),
        triggering_penalty_usd=('triggering_penalty_usd', 'sum'),
        forgone_reward_usd=('forgone_reward_usd', 'sum'),
        removed_orders=('removed_orders', 'sum'),
        coverage_at_risk_orders_ub=('coverage_at_risk_orders_ub', 'sum'),
        future_penalty_diag_usd=('future_penalty_diag_usd', 'sum'),
    ).reset_index()
    g.insert(0, 'schedule', name)
    rows.append(g)

sit_sweep = pd.concat(rows, ignore_index=True)
display(sit_sweep.round(2))
plot_sit_out(detail["1 auction (flat)"])

## 9. Within-solver paired comparison (cross-chain)
Same solver on different-cap chains, controls for solver skill/infra/flow, cross-chain confounder. Needs ≥2 chains.

In [ ]:
def within_solver(df, min_n=None):
    g = (
        df.groupby(['solver_name', 'blockchain'], dropna=False)
        .agg(
            n=('order_uid', 'size'),
            revert_rate=('reverted', 'mean'),
            med_gross_pnl_usd=('gross_execution_pnl_usd', 'median'),
            med_net_pnl_usd=('net_mechanism_pnl_usd', 'median'),
            med_markout_bps=('markout_bps', 'median'),
            penalty_cap_usd=('penalty_cap_usd', 'median'),
        )
        .reset_index()
    )
    if min_n is not None:
        g = g[g['n'] >= min_n].copy()
    multi = g.groupby('solver_name')['blockchain'].transform('nunique') > 1
    return (
        g[multi]
        .sort_values(['solver_name', 'penalty_cap_usd'])
        .reset_index(drop=True)
    )
def plot_within_solver(tbl):
    if tbl.empty or tbl['blockchain'].nunique() < 2:
        print('Need solvers active on ≥2 chains.')
        return
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    solvers = list(tbl['solver_name'].unique())
    cmap = {sv: col for sv, col in zip(solvers, plt.cm.tab20(np.linspace(0, 1, max(len(solvers), 1))))}
    for sv, g in tbl.groupby('solver_name'):
        g = g.sort_values('penalty_cap_usd')
        ax[0].plot(
            g['penalty_cap_usd'],
            g['revert_rate'] * 100,
            marker='o',
            color=cmap[sv],
            label=sv,
        )
        ax[1].plot(
            g['penalty_cap_usd'],
            g['med_gross_pnl_usd'],
            marker='o',
            color=cmap[sv],
            label=sv,
        )
        ax[2].plot(
            g['penalty_cap_usd'],
            g['med_markout_bps'],
            marker='o',
            color=cmap[sv],
            label=sv,
        )
    ax[0].set_ylabel('revert rate (%)')
    ax[1].set_ylabel('median gross P&L (USD)')
    ax[2].set_ylabel('median markout (bps)')
    ax[0].set_title('Within-solver revert rate vs cap')
    ax[1].set_title('Within-solver P&L vs cap')
    ax[2].set_title('Within-solver markout vs cap')
    for a in ax:
        a.set_xlabel('penalty cap (USD)')

    handles, labels = ax[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
               fontsize=7, ncol=1 + len(solvers) // 20, title='solver')
    plt.tight_layout(rect=(0, 0, 0.98, 1))
    plt.show()
T11 = within_solver(DF)
display(T11.head(20).round({
    'revert_rate': 4,
    'med_gross_pnl_usd': 4,
    'med_net_pnl_usd': 4,
    'med_markout_bps': 2,
    'penalty_cap_usd': 4,
}))

plot_within_solver(T11)

## 10. Cap clipping vs markout
Compare cap clipping on reverted auction-solver outcomes with settled-execution markout at the chain and solver level. Both cap-binding frequency and forgiven-penalty magnitude, then repeats the correlations within fixed order-size buckets as a size-control diagnostic. Group level association (X on reverts & Y on settled)

In [ ]:
def _cap_revert_sample(rps):
    return rps[
        rps['reverted']
        & (rps['penalty_uncapped_usd'] > 0)
        & (rps['penalty_cap_usd'] > 0)
        & (rps['penalty_usd'].notna())
    ].copy()


def cap_vs_markout(df, unit, min_reverts=50, min_settled=50, size_bucket=False):
    rps = reward_per_auction_solver(df)
    keys = [unit, 'size_fixed'] if size_bucket else [unit]

    rev = _cap_revert_sample(rps)

    if size_bucket:
        rev = add_fixed_buckets(rev)

    X = (
        rev.groupby(keys, observed=True)
        .agg(
            cap_binds_rate=('cap_binds', 'mean'),
            n_reverts=('cap_binds', 'size')
        )
        .reset_index()
    )

    st = df[~df['reverted']].copy()

    if size_bucket:
        st = add_fixed_buckets(st)

    Y = (
        st.groupby(keys, observed=True)
        .apply(
            lambda g: pd.Series({
                'med_markout_bps': g['markout_bps'].median(),
                'n_settled': g['markout_bps'].notna().sum(),
            }),
            include_groups=False
        )
        .reset_index()
    )

    m = X.merge(Y, on=keys, how='inner')
    m = m[
        (m['n_reverts'] >= min_reverts)
        & (m['n_settled'] >= min_settled)
    ].copy()

    return m


def plot_cap_vs_markout(tbl, unit, ycol='med_markout_bps', ylabel='median markout (bps)', label_solvers=False):
    if len(tbl) < 3:
        print(f'Need ≥3 units with enough sample (have {len(tbl)}).')
        return

    fig, ax = plt.subplots(figsize=(11 if label_solvers else 9, 7 if label_solvers else 6))

    chains = list(tbl['blockchain'].unique()) if 'blockchain' in tbl else ['all']
    cmap = {
        c: col
        for c, col in zip(
            chains,
            plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1)))
        )
    }

    color_key = 'blockchain' if 'blockchain' in tbl else None
    sizes = 40 + 200 * (tbl['n_settled'] / tbl['n_settled'].max())

    ax.scatter(
        tbl['cap_binds_rate'] * 100,
        tbl[ycol],
        s=sizes,
        c=[
            cmap[tbl[color_key].iloc[i]]
            if color_key else '#4f81bd'
            for i in range(len(tbl))
        ],
        alpha=.75,
        edgecolor='k',
        linewidth=.5
    )

    if unit == 'blockchain':
        for _, r in tbl.iterrows():
            ax.annotate(
                r['blockchain'],
                (r['cap_binds_rate'] * 100, r[ycol]),
                textcoords='offset points',
                xytext=(5, 5),
                fontsize=8
            )

    if label_solvers and 'solver_name' in tbl:
        for _, r in tbl.iterrows():
            ax.annotate(
                str(r['solver_name']),
                (r['cap_binds_rate'] * 100, r[ycol]),
                textcoords='offset points',
                xytext=(5, 5),
                fontsize=7
            )

    rho, p = stats.spearmanr(tbl['cap_binds_rate'], tbl[ycol])

    ax.set(
        xlabel='cap-binding rate among reverts (%)',
        ylabel=ylabel,
        title=f'Cap-binding vs markout, per {unit}  (Spearman ρ={rho:.2f}, p={p:.3f})'
    )

    ax.axhline(0, color='k', lw=.6, alpha=.5)
    plt.tight_layout()
    plt.show()

    return rho, p


def forgiven_vs_markout(df, unit, min_reverts=50, min_settled=50, size_bucket=False):
    rps = reward_per_auction_solver(df).copy()
    keys = [unit, 'size_fixed'] if size_bucket else [unit]

    rev = _cap_revert_sample(rps)

    rev['penalty_forgiven_usd'] = (
        rev['penalty_uncapped_usd'] - rev['penalty_usd']
    ).clip(lower=0)

    if size_bucket:
        rev = add_fixed_buckets(rev)

    X = (
        rev.groupby(keys, observed=True)
        .agg(
            n_reverts=('reverted', 'size'),
            cap_binds_rate=('cap_binds', 'mean'),
            uncapped_sum_usd=('penalty_uncapped_usd', 'sum'),
            paid_penalty_sum_usd=('penalty_usd', 'sum'),
            forgiven_sum_usd=('penalty_forgiven_usd', 'sum'),
            forgiven_median_usd=('penalty_forgiven_usd', 'median'),
            reverted_volume_usd=('volume_usd', 'sum'),
        )
        .reset_index()
    )

    X['forgiven_share_of_uncapped'] = (
        X['forgiven_sum_usd'] / X['uncapped_sum_usd']
    ).replace([np.inf, -np.inf], np.nan)

    X['forgiven_bps_of_reverted_volume'] = (
        X['forgiven_sum_usd'] / X['reverted_volume_usd']
    ).replace([np.inf, -np.inf], np.nan) * 1e4

    st = df[~df['reverted']].copy()

    if size_bucket:
        st = add_fixed_buckets(st)

    Y = (
        st.groupby(keys, observed=True)
        .apply(
            lambda g: pd.Series({
                'med_markout_bps': g['markout_bps'].median(),
                'n_settled': g['markout_bps'].notna().sum(),
            }),
            include_groups=False
        )
        .reset_index()
    )

    m = X.merge(Y, on=keys, how='inner')
    m = m[
        (m['n_reverts'] >= min_reverts)
        & (m['n_settled'] >= min_settled)
    ].copy()

    return m


def plot_forgiven_vs_markout(
    tbl,
    unit,
    xcol,
    xlabel,
    x_as_percent=False,
    ycol='med_markout_bps',
    ylabel='median markout (bps)',
    label_solvers=False,
):
    plot_tbl = (
        tbl.replace([np.inf, -np.inf], np.nan)
        .dropna(subset=[xcol, ycol])
        .copy()
    )

    if len(plot_tbl) < 3:
        print(f'Need ≥3 finite observations after filtering (have {len(plot_tbl)}).')
        return

    fig, ax = plt.subplots(figsize=(11 if label_solvers else 9, 7 if label_solvers else 6))

    x = plot_tbl[xcol] * 100 if x_as_percent else plot_tbl[xcol]
    sizes = 40 + 200 * (plot_tbl['n_settled'] / plot_tbl['n_settled'].max())

    if 'blockchain' in plot_tbl:
        chains = list(plot_tbl['blockchain'].unique())
        cmap = {
            c: col
            for c, col in zip(
                chains,
                plt.cm.tab10(np.linspace(0, 1, max(len(chains), 1)))
            )
        }
        colors = [cmap[c] for c in plot_tbl['blockchain']]
    else:
        colors = '#4f81bd'

    ax.scatter(
        x,
        plot_tbl[ycol],
        s=sizes,
        c=colors,
        alpha=.75,
        edgecolor='k',
        linewidth=.5,
    )

    if unit == 'blockchain':
        for _, r in plot_tbl.iterrows():
            x_val = r[xcol] * 100 if x_as_percent else r[xcol]
            ax.annotate(
                r['blockchain'],
                (x_val, r[ycol]),
                textcoords='offset points',
                xytext=(5, 5),
                fontsize=8,
            )

    if label_solvers and 'solver_name' in plot_tbl:
        for _, r in plot_tbl.iterrows():
            x_val = r[xcol] * 100 if x_as_percent else r[xcol]
            ax.annotate(
                str(r['solver_name']),
                (x_val, r[ycol]),
                textcoords='offset points',
                xytext=(5, 5),
                fontsize=7,
            )

    rho, p = stats.spearmanr(plot_tbl[xcol], plot_tbl[ycol])

    ax.set(
        xlabel=xlabel,
        ylabel=ylabel,
        title=f'{xlabel} vs markout, per {unit}  (Spearman ρ={rho:.2f}, p={p:.3f})',
    )

    ax.axhline(0, color='k', lw=.6, alpha=.5)
    plt.tight_layout()
    plt.show()

    return rho, p


def size_stratified_cap_table(strat):
    rows = []

    for b in FIXED_LABELS:
        sub = strat[strat['size_fixed'] == b]

        if len(sub) >= 4:
            rho, p = stats.spearmanr(
                sub['cap_binds_rate'],
                sub['med_markout_bps']
            )
            rows.append({
                'size_band': b,
                'n_units': len(sub),
                'spearman_rho': round(rho, 2),
                'p_value': round(p, 3),
            })
        else:
            rows.append({
                'size_band': b,
                'n_units': len(sub),
                'spearman_rho': np.nan,
                'p_value': np.nan,
            })

    return pd.DataFrame(rows)


def size_stratified_forgiven_table(strat):
    rows = []

    for b in FIXED_LABELS:
        sub = (
            strat[strat['size_fixed'] == b]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )

        row = {
            'size_band': b,
            'n_units': len(sub),
        }

        metrics = [
            (
                'forgiven_share_of_uncapped',
                'rho_forgiven_share',
                'p_forgiven_share',
            ),
            (
                'forgiven_bps_of_reverted_volume',
                'rho_forgiven_bps_volume',
                'p_forgiven_bps_volume',
            ),
        ]

        for xcol, rho_col, p_col in metrics:
            clean = sub[[xcol, 'med_markout_bps']].dropna()

            if len(clean) >= 4:
                rho, p = stats.spearmanr(
                    clean[xcol],
                    clean['med_markout_bps'],
                )
                row[rho_col] = round(rho, 2)
                row[p_col] = round(p, 3)
            else:
                row[rho_col] = np.nan
                row[p_col] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)

T10_chain = cap_vs_markout(DF, 'blockchain', min_reverts=MIN_REVERTS_CHAIN, min_settled=MIN_SETTLED_CHAIN)
print('=== cap-binding vs markout, per chain ===')
display(T10_chain.round(3))
plot_cap_vs_markout(
    T10_chain,
    'blockchain',
    'med_markout_bps',
    'median markout (bps)'
)

T10_solver = cap_vs_markout(DF, 'solver_name',  min_reverts=MIN_REVERTS, min_settled=MIN_SETTLED)
print(f'=== cap-binding vs markout, per solver (n={len(T10_solver)}) ===')
display(T10_solver.sort_values('cap_binds_rate', ascending=False).round(3))
plot_cap_vs_markout(
    T10_solver,
    'solver_name',
    'med_markout_bps',
    'median markout (bps)',
    label_solvers=True,
)

cap_strat = cap_vs_markout(
    DF,
    'solver_name',
    min_reverts=MIN_REVERTS_CHAIN, min_settled=MIN_SETTLED_CHAIN, size_bucket=True
)

print('=== size-stratified: cap-binding vs markout ===')
display(size_stratified_cap_table(cap_strat))

F10_chain = forgiven_vs_markout(
    DF,
    'blockchain',
    min_reverts=MIN_REVERTS_CHAIN, min_settled=MIN_SETTLED_CHAIN
)

print('=== forgiven penalty vs markout, per chain ===')
display(
    F10_chain.round({
        'cap_binds_rate': 3,
        'uncapped_sum_usd': 2,
        'paid_penalty_sum_usd': 2,
        'forgiven_sum_usd': 2,
        'forgiven_median_usd': 4,
        'forgiven_share_of_uncapped': 3,
        'forgiven_bps_of_reverted_volume': 3,
        'med_markout_bps': 3,
    })
)

plot_forgiven_vs_markout(
    F10_chain,
    'blockchain',
    xcol='forgiven_share_of_uncapped',
    xlabel='forgiven penalty / uncapped penalty (%)',
    x_as_percent=True,
)

plot_forgiven_vs_markout(
    F10_chain,
    'blockchain',
    xcol='forgiven_bps_of_reverted_volume',
    xlabel='forgiven penalty (bps of reverted volume)',
)

F10_solver = forgiven_vs_markout(
    DF,
    'solver_name',
    min_reverts=MIN_REVERTS, min_settled=MIN_SETTLED
)

print(f'=== forgiven penalty vs markout, per solver (n={len(F10_solver)}) ===')
display(
    F10_solver
    .sort_values('forgiven_share_of_uncapped', ascending=False)
    .round({
        'cap_binds_rate': 3,
        'uncapped_sum_usd': 2,
        'paid_penalty_sum_usd': 2,
        'forgiven_sum_usd': 2,
        'forgiven_median_usd': 4,
        'forgiven_share_of_uncapped': 3,
        'forgiven_bps_of_reverted_volume': 3,
        'med_markout_bps': 3,
    })
)

plot_forgiven_vs_markout(
    F10_solver,
    'solver_name',
    xcol='forgiven_share_of_uncapped',
    xlabel='forgiven penalty / uncapped penalty (%)',
    x_as_percent=True,
    label_solvers=True,
)

plot_forgiven_vs_markout(
    F10_solver,
    'solver_name',
    xcol='forgiven_bps_of_reverted_volume',
    xlabel='forgiven penalty (bps of reverted volume)',
    label_solvers=True,
)

forgiven_strat = forgiven_vs_markout(
    DF,
    'solver_name',
    min_reverts=MIN_REVERTS_CHAIN, min_settled=MIN_SETTLED_CHAIN,
    size_bucket=True,
)

print('=== size-stratified: forgiven penalty vs markout ===')
display(size_stratified_forgiven_table(forgiven_strat))

## 11. Cap / winning surplus vs markout and execution time
Empirical test of the theoretical curve:

- x-axis = penalty cap / winning surplus
- y-axis 1 = median markout
- y-axis 2 = median execution time / time-to-happy-moo


This uses binned medians per chain instead of a fitted correlation line, because the predicted relationship is non-linear / threshold-shaped.

In [ ]:
CAP_SURPLUS_MIN_OBS_PER_BIN = 50
TARGET_SOLVER = None
# TARGET_SOLVER = 'Quasi'   # set this to run only for a single solver
SOLVER_COL = 'solver_name'
SOLVER_MATCH_EXACT = True
CAP_SURPLUS_BINS = [
    0,
    0.01,
    0.03,
    0.10,
    0.30,
    1.00,
    3.00,
    10.00,
    30.00,
    100.00,
    np.inf,
]
CAP_SURPLUS_LABELS = [
    '<0.01x',
    '0.01-0.03x',
    '0.03-0.10x',
    '0.10-0.30x',
    '0.30-1x',
    '1-3x',
    '3-10x',
    '10-30x',
    '30-100x',
    '>100x',
]

def filter_solver(df, solver_name=None, solver_col=SOLVER_COL, exact=SOLVER_MATCH_EXACT):
    if solver_name is None:
        return df.copy(), 'all solvers'

    if solver_col not in df.columns:
        raise KeyError(f"{solver_col!r} not found in df columns.")

    d = df.copy()

    solver_text = d[solver_col].astype(str).str.lower()
    target = str(solver_name).lower()

    if exact:
        mask = solver_text == target
    else:
        mask = solver_text.str.contains(target, regex=False, na=False)

    matched = d.loc[mask, solver_col].dropna().astype(str).value_counts()

    print(f"=== solver filter: {solver_name!r} ===")
    print(f"matched rows: {mask.sum():,} / {len(d):,}")

    if len(matched):
        print("matched solver labels:")
        display(
            matched
            .head(20)
            .rename('rows')
            .reset_index()
            .rename(columns={'index': solver_col})
        )
    else:
        print("No solver labels matched. Check available solver labels:")
        display(
            d[solver_col]
            .dropna()
            .astype(str)
            .value_counts()
            .head(50)
            .rename('rows')
            .reset_index()
            .rename(columns={'index': solver_col})
        )

    return d[mask].copy(), str(solver_name)

def cap_surplus_sample(
    df,
    surplus_col='observed_score_usd',
    solver_name=TARGET_SOLVER,
    solver_col=SOLVER_COL,
    solver_exact=SOLVER_MATCH_EXACT,
):
    d0, solver_label = filter_solver(
        df,
        solver_name=solver_name,
        solver_col=solver_col,
        exact=solver_exact,
    )

    if surplus_col not in d0.columns:
        raise KeyError(
            f"{surplus_col!r} not found. Expected winning surplus proxy column. "
            "Use observed_score_usd if prepare() has already run."
        )

    d = d0[d0['settled'] == True].copy()

    d['winning_surplus_usd'] = pd.to_numeric(d[surplus_col], errors='coerce')
    d['penalty_cap_usd'] = pd.to_numeric(d['penalty_cap_usd'], errors='coerce')
    d['markout_bps'] = pd.to_numeric(d['markout_bps'], errors='coerce')
    d['order_execution_time_s'] = pd.to_numeric(d['seconds_since_created'], errors='coerce')

    d['cap_over_winning_surplus'] = (
        d['penalty_cap_usd'] / d['winning_surplus_usd']
    ).replace([np.inf, -np.inf], np.nan)

    before = len(d)

    d = d[
        d['cap_over_winning_surplus'].notna()
        & (d['cap_over_winning_surplus'] > 0)
        & d['winning_surplus_usd'].notna()
        & (d['winning_surplus_usd'] > 0)
        & d['penalty_cap_usd'].notna()
        & (d['penalty_cap_usd'] > 0)
    ].copy()

    d['_solver_filter_label'] = solver_label

    print(
        f"cap/surplus sample for {solver_label}: {before:,} settled rows -> {len(d):,} rows "
        f"after requiring positive cap and positive winning surplus"
    )

    return d

def cap_surplus_binned_by_chain(
    df,
    surplus_col='observed_score_usd',
    min_obs=CAP_SURPLUS_MIN_OBS_PER_BIN,
    solver_name=TARGET_SOLVER,
    solver_col=SOLVER_COL,
    solver_exact=SOLVER_MATCH_EXACT,
):
    d = cap_surplus_sample(
        df,
        surplus_col=surplus_col,
        solver_name=solver_name,
        solver_col=solver_col,
        solver_exact=solver_exact,
    )

    d['cap_surplus_bin'] = pd.cut(
        d['cap_over_winning_surplus'],
        bins=CAP_SURPLUS_BINS,
        labels=CAP_SURPLUS_LABELS,
        include_lowest=True,
    )

    binned = (
        d
        .dropna(subset=['cap_surplus_bin'])
        .groupby(['blockchain', 'cap_surplus_bin'], observed=True)
        .agg(
            n=('order_uid', 'size'),
            n_markout=('markout_bps', lambda s: s.notna().sum()),
            n_execution_time=('order_execution_time_s', lambda s: s.notna().sum()),
            x_median=('cap_over_winning_surplus', 'median'),
            x_p25=('cap_over_winning_surplus', lambda s: s.quantile(0.25)),
            x_p75=('cap_over_winning_surplus', lambda s: s.quantile(0.75)),
            penalty_cap_usd=('penalty_cap_usd', 'median'),
            winning_surplus_usd=('winning_surplus_usd', 'median'),
            med_markout_bps=('markout_bps', 'median'),
            med_execution_time_s=('order_execution_time_s', 'median'),
            volume_usd=('volume_usd', 'sum'),
            median_size_usd=('size_usd', 'median'),
        )
        .reset_index()
    )

    binned = binned[
        (binned['n_markout'] >= min_obs)
        & (binned['n_execution_time'] >= min_obs)
        & binned['x_median'].notna()
    ].copy()

    binned['_solver_filter_label'] = 'all solvers' if solver_name is None else str(solver_name)

    return binned.sort_values(['blockchain', 'x_median'])

def plot_cap_surplus_binned_curves(binned):
    if binned.empty:
        print("No binned data to plot.")
        return
    
    solver_label = (
        binned['_solver_filter_label'].iloc[0]
        if '_solver_filter_label' in binned.columns and len(binned)
        else ''
    )

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            'Cap / winning surplus vs markout',
            'Cap / winning surplus vs time to execution',
        ],
        horizontal_spacing=0.10,
    )

    outcomes = [
        ('med_markout_bps', 'Median markout', 'bps'),
        ('med_execution_time_s', 'Median time to execution', 'seconds'),
    ]

    chains = sorted(binned['blockchain'].dropna().unique())

    plotly_colors = [
        '#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
        '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52',
    ]
    chain_color = {
        ch: plotly_colors[i % len(plotly_colors)]
        for i, ch in enumerate(chains)
    }

    for col_idx, (ycol, ylabel, unit) in enumerate(outcomes, start=1):
        for ch in chains:
            g = (
                binned[binned['blockchain'] == ch]
                .replace([np.inf, -np.inf], np.nan)
                .dropna(subset=['x_median', ycol])
                .sort_values('x_median')
                .copy()
            )

            if g.empty:
                continue

            fig.add_trace(
                go.Scatter(
                    x=g['x_median'],
                    y=g[ycol],
                    mode='lines+markers',
                    name=ch,
                    legendgroup=ch,
                    showlegend=(col_idx == 1),
                    marker=dict(
                        size=8,
                        color=chain_color[ch],
                        line=dict(width=0.7, color='black'),
                    ),
                    line=dict(
                        width=2,
                        color=chain_color[ch],
                    ),
                    customdata=np.stack(
                        [
                            g['blockchain'],
                            g['cap_surplus_bin'].astype(str),
                            g['n'],
                            g['n_markout'],
                            g['n_execution_time'],
                            g['penalty_cap_usd'],
                            g['winning_surplus_usd'],
                            g['median_size_usd'],
                            g['volume_usd'],
                        ],
                        axis=-1,
                    ),
                    hovertemplate=(
                        '<b>%{customdata[0]}</b><br>'
                        'cap / winning surplus: %{x:.3f}x<br>'
                        'bin: %{customdata[1]}<br>'
                        f'{ylabel}: %{{y:,.2f}} {unit}<br>'
                        'rows: %{customdata[2]:,.0f}<br>'
                        'markout obs: %{customdata[3]:,.0f}<br>'
                        'execution-time obs: %{customdata[4]:,.0f}<br>'
                        'median penalty cap: $%{customdata[5]:,.2f}<br>'
                        'median winning surplus: $%{customdata[6]:,.2f}<br>'
                        'median size: $%{customdata[7]:,.2f}<br>'
                        'volume: $%{customdata[8]:,.0f}'
                        '<extra></extra>'
                    ),
                ),
                row=1,
                col=col_idx,
            )

        fig.add_vline(
            x=1,
            line_width=1.5,
            line_dash='dash',
            line_color='black',
            row=1,
            col=col_idx,
        )

        fig.add_annotation(
            x=1,
            y=1.02,
            xref=f'x{col_idx}' if col_idx > 1 else 'x',
            yref='paper',
            text='cap = winning surplus',
            showarrow=False,
            font=dict(size=11),
            row=1,
            col=col_idx,
        )

        fig.update_xaxes(
            title_text='Penalty cap / winning surplus',
            type='log',
            tickmode='array',
            tickvals=[0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
            ticktext=['0.01x', '0.03x', '0.1x', '0.3x', '1x', '3x', '10x', '30x', '100x'],
            row=1,
            col=col_idx,
        )

        fig.update_yaxes(
            title_text=f'{ylabel} ({unit})',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='black',
            row=1,
            col=col_idx,
        )

    fig.update_layout(
        title=f'Empirical shape for {solver_label}: cap / winning surplus vs markout and execution time',
        width=1550,
        height=650,
        legend_title_text='chain',
        template='plotly_white',
    )

    fig.show()


CAP_SURPLUS_BINNED = cap_surplus_binned_by_chain(
    DF,
    surplus_col='observed_score_usd',
    min_obs=CAP_SURPLUS_MIN_OBS_PER_BIN,
    solver_name=TARGET_SOLVER,
    solver_col=SOLVER_COL,
    solver_exact=SOLVER_MATCH_EXACT,
)

print("=== binned cap / winning surplus outcomes ===")
display(
    CAP_SURPLUS_BINNED[
        [
            'blockchain',
            'cap_surplus_bin',
            'n',
            'n_markout',
            'n_execution_time',
            'x_median',
            'penalty_cap_usd',
            'winning_surplus_usd',
            'med_markout_bps',
            'med_execution_time_s',
            'median_size_usd',
            'volume_usd',
        ]
    ].round({
        'x_median': 4,
        'penalty_cap_usd': 4,
        'winning_surplus_usd': 4,
        'med_markout_bps': 2,
        'med_execution_time_s': 2,
        'median_size_usd': 2,
        'volume_usd': 2,
    })
)
print("=== bin coverage by chain ===")
display(
    CAP_SURPLUS_BINNED
    .groupby('blockchain')
    .agg(
        bins=('cap_surplus_bin', 'nunique'),
        total_rows=('n', 'sum'),
        x_min=('x_median', 'min'),
        x_max=('x_median', 'max'),
        med_markout_bps_min=('med_markout_bps', 'min'),
        med_markout_bps_max=('med_markout_bps', 'max'),
        med_execution_time_s_min=('med_execution_time_s', 'min'),
        med_execution_time_s_max=('med_execution_time_s', 'max'),
    )
    .round({
        'x_min': 4,
        'x_max': 4,
        'med_markout_bps_min': 2,
        'med_markout_bps_max': 2,
        'med_execution_time_s_min': 2,
        'med_execution_time_s_max': 2,
    })
)
plot_cap_surplus_binned_curves(CAP_SURPLUS_BINNED)

## 12. Direct penalty P&L impact via consistency redistribution
Objective, per solver:
- bar 1: penalties paid in USD
- bar 2: weekly consistency share × weekly sum of penalties

This isolates the penalty channel only: `penalty_reimbursement_s = sum_over_chain_weeks(weekly_solver_share_s,c,w * weekly_total_penalties_c,w)`

The redistribution is chain-week local: penalties on a chain are redistributed using consistency shares from the same chain and same accounting week.

In [ ]:
BID_QUERY_ID_11 = 7405278
CACHE_DIR_11 = "."
SUPPORTED_CHAINS_11 = {'base', 'ethereum'}
ANONYMIZE_SOLVERS_11 = True
TOP_N_SOLVERS_11 = 20

SOLVER_NAME_MAP_11 = {
    'sector': 'Sector_Finance',
    'sector_finance': 'Sector_Finance',
}

def canonical_solver_label_11(x):
    if pd.isna(x):
        return x

    s = str(x).strip()
    key = (
        s.lower()
        .replace(' ', '_')
        .replace('-', '_')
    )

    return SOLVER_NAME_MAP_11.get(key, s)


ALL_CHAINS_11 = sorted(DF['blockchain'].dropna().unique().tolist())
CHAINS_11 = [ch for ch in ALL_CHAINS_11 if ch in SUPPORTED_CHAINS_11]
SKIPPED_CHAINS_11 = [ch for ch in ALL_CHAINS_11 if ch not in SUPPORTED_CHAINS_11]

if SKIPPED_CHAINS_11:
    print("skipping chains not supported by Dune query:", SKIPPED_CHAINS_11)

if not CHAINS_11:
    raise RuntimeError("No supported chains found in DF. This Dune query only supports base and ethereum.")

DF_11 = DF[DF['blockchain'].isin(CHAINS_11)].copy()

START_TIME_11 = '2026-04-14 00:00:00'
END_TIME_11 = '2026-06-23 00:00:00'

lo_11 = pd.Timestamp(START_TIME_11, tz='UTC')
hi_11 = pd.Timestamp(END_TIME_11, tz='UTC')

def local_bid_csv_for_chain(ch):
    candidates = sorted(
        glob.glob(os.path.join(CACHE_DIR_11, f"consistency_bids_{ch}*.csv")),
        key=os.path.getmtime,
        reverse=True,
    )

    for path in candidates:
        try:
            sample = pd.read_csv(path, usecols=['accounting_week_start'], low_memory=False)
            periods = (
                pd.to_datetime(sample['accounting_week_start'], utc=True, errors='coerce')
                .dt.tz_convert(None)
                .dt.normalize()
                .dropna()
            )

            if len(periods) == 0:
                continue

            print(f"local csv coverage for {ch}: {periods.min()} to {periods.max()}")
            return path

        except Exception as e:
            print(f"could not validate local csv for {ch}, skipping: {path} ({e})")

    return None


dune = None
bid_frames = []

for ch in CHAINS_11:
    cache_path = local_bid_csv_for_chain(ch)

    if cache_path is not None:
        print(f"using local csv for {ch}: {cache_path}")
        bids_ch = pd.read_csv(cache_path, low_memory=False)
    else:
        if dune is None:
            vals = dotenv_values('/workspaces/penalty-research/.env')
            key = vals.get('DUNE_API_KEY')
            if not key:
                raise RuntimeError("Set DUNE_API_KEY in your environment or .env file")
            dune = DuneClient(key)

        cache_key = f"{BID_QUERY_ID_11}|{ch}|{START_TIME_11}|{END_TIME_11}"
        slug = hashlib.md5(cache_key.encode()).hexdigest()[:10]
        cache_path = os.path.join(CACHE_DIR_11, f"consistency_bids_{ch}_{slug}.csv")

        print(f"running Dune query {BID_QUERY_ID_11} for {ch}")
        query = QueryBase(
            query_id=BID_QUERY_ID_11,
            params=[
                QueryParameter.text_type(name='blockchain', value=ch),
                QueryParameter.date_type(name='start_time', value=START_TIME_11),
                QueryParameter.date_type(name='end_time', value=END_TIME_11),
            ],
        )

        bids_ch = dune.run_query_dataframe(query)
        bids_ch.to_csv(cache_path, index=False)
        print(f"cached {len(bids_ch):,} rows -> {cache_path}")

    if len(bids_ch) == 0:
        print(f"no bid rows for {ch}, skipping")
        continue

    bids_ch['blockchain'] = ch
    bid_frames.append(bids_ch)

if not bid_frames:
    raise RuntimeError("No bid data fetched for any chain.")

BIDS_11 = pd.concat(bid_frames, ignore_index=True)
print(f"loaded {len(BIDS_11):,} bid rows")

required_cols_11 = [
    'auction_time',
    'accounting_week_start',
    'auction_id',
    'order_uid',
    'solver',
    'solver_name',
    'is_winner',
    'filtered_out',
    'settled',
    'relative_surplus_vs_winner',
]

missing_cols_11 = [c for c in required_cols_11 if c not in BIDS_11.columns]
if missing_cols_11:
    raise KeyError(f"Dune bid data missing required columns: {missing_cols_11}")

bids = BIDS_11.copy()
bids = bids.replace(['<nil>', '<null>', 'nan', 'None', ''], np.nan)

bids['auction_time'] = pd.to_datetime(bids['auction_time'], utc=True, errors='coerce')

bids['period'] = (
    pd.to_datetime(bids['accounting_week_start'], utc=True, errors='coerce')
    .dt.tz_convert(None)
    .dt.normalize()
)

bids = bids.dropna(subset=['auction_time', 'period'])

for c in ['is_winner', 'filtered_out', 'settled']:
    bids[c] = (
        bids[c]
        .astype(str)
        .str.lower()
        .map({'true': True, 'false': False, '1': True, '0': False})
        .fillna(False)
        .astype(bool)
    )

bids['relative_surplus_vs_winner'] = (
    pd.to_numeric(bids['relative_surplus_vs_winner'], errors='coerce')
    .clip(lower=0)
)

bids['solver_addr'] = bids['solver'].astype(str).str.lower().str.strip()

bids['solver_label'] = (
    bids['solver_name']
    .where(bids['solver_name'].notna(), bids['solver'])
    .map(canonical_solver_label_11)
    .astype(str)
)

bids = bids[
    (bids['auction_time'] >= lo_11)
    & (bids['auction_time'] < hi_11)
].copy()

best = (
    bids[~bids['filtered_out']]
    .dropna(subset=['relative_surplus_vs_winner'])
    .sort_values('relative_surplus_vs_winner')
    .drop_duplicates(
        ['blockchain', 'period', 'solver_addr', 'auction_id', 'order_uid'],
        keep='last'
    )
    .copy()
)

trade_total = (
    best
    .groupby(['blockchain', 'period', 'auction_id', 'order_uid'])['relative_surplus_vs_winner']
    .transform('sum')
)

best['relative_surplus_share'] = (
    best['relative_surplus_vs_winner']
    / trade_total.replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

surplus_component = (
    best
    .groupby(['blockchain', 'period', 'solver_addr'])
    .agg(surplus_share_sum=('relative_surplus_share', 'sum'))
    .reset_index()
)

wins = (
    bids[bids['is_winner']]
    .drop_duplicates(['blockchain', 'period', 'solver_addr', 'auction_id', 'order_uid'])
    .copy()
)

success = (
    wins
    .groupby(['blockchain', 'period', 'solver_addr'])
    .agg(
        won_orders=('order_uid', 'size'),
        settled_orders=('settled', 'sum'),
    )
    .reset_index()
)

success['success_rate'] = (
    success['settled_orders'] / success['won_orders'].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

if (success['success_rate'] > 1 + 1e-12).any():
    display(success[success['success_rate'] > 1 + 1e-12])
    raise RuntimeError("success_rate > 1 found")

metric = surplus_component.merge(
    success[['blockchain', 'period', 'solver_addr', 'success_rate']],
    on=['blockchain', 'period', 'solver_addr'],
    how='left'
)

metric['success_rate'] = metric['success_rate'].fillna(0)
metric['consistency_metric'] = metric['surplus_share_sum'] * metric['success_rate']

metric['weekly_metric_total'] = (
    metric
    .groupby(['blockchain', 'period'])['consistency_metric']
    .transform('sum')
)

metric['weekly_solver_share'] = (
    metric['consistency_metric']
    / metric['weekly_metric_total'].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

solver_labels = (
    bids
    .groupby(['blockchain', 'solver_addr'])['solver_label']
    .first()
    .reset_index()
)

metric = metric.merge(
    solver_labels,
    on=['blockchain', 'solver_addr'],
    how='left'
)

display(
    metric
    .groupby(['blockchain', 'period'])['weekly_solver_share']
    .sum()
    .groupby('blockchain')
    .agg(
        weeks='count',
        avg_share_sum='mean',
        min_share_sum='min',
        max_share_sum='max',
    )
    .round(6)
)

rps = reward_per_auction_solver(DF_11).copy()

time_lookup = (
    DF_11.groupby(['blockchain', 'auction_id', 'solver'], dropna=False)
      .agg(
          auction_timestamp=('auction_timestamp', 'first'),
          solver_name=('solver_name', 'first'),
      )
      .reset_index()
)

rps = rps.merge(
    time_lookup,
    on=['blockchain', 'auction_id', 'solver'],
    how='left',
    suffixes=('', '_from_df')
)

if 'solver_name_from_df' in rps.columns:
    rps['solver_name'] = rps['solver_name'].where(
        rps['solver_name'].notna(),
        rps['solver_name_from_df']
    )
    rps = rps.drop(columns=['solver_name_from_df'])

rps['solver_name'] = rps['solver_name'].map(canonical_solver_label_11)

rps['auction_timestamp'] = pd.to_datetime(rps['auction_timestamp'], utc=True, errors='coerce')

rps = rps[
    (rps['auction_timestamp'] >= lo_11)
    & (rps['auction_timestamp'] < hi_11)
].copy()

rps['_day'] = rps['auction_timestamp'].dt.tz_convert(None).dt.floor('D')

rps['period'] = (
    rps['_day']
    - pd.to_timedelta((rps['_day'].dt.dayofweek - 1) % 7, unit='D')
)

rps['period'] = pd.to_datetime(rps['period'], errors='coerce').dt.normalize()
rps['solver_addr'] = rps['solver'].astype(str).str.lower().str.strip()
rps['penalty_usd'] = rps['penalty_usd'].fillna(0)

solver_weekly_penalties = (
    rps
    .groupby(['blockchain', 'period', 'solver_addr', 'solver_name'], dropna=False)
    .agg(
        penalties_paid_usd=('penalty_usd', 'sum'),
        reverts=('reverted', 'sum'),
    )
    .reset_index()
)

weekly_penalties = (
    solver_weekly_penalties
    .groupby(['blockchain', 'period'])
    .agg(weekly_sum_of_penalty=('penalties_paid_usd', 'sum'))
    .reset_index()
)

display(
    weekly_penalties
    .groupby('blockchain')
    .agg(
        weeks=('period', 'nunique'),
        total_penalties_usd=('weekly_sum_of_penalty', 'sum'),
    )
    .round(4)
)

metric['period'] = pd.to_datetime(metric['period'], errors='coerce').dt.normalize()
weekly_penalties['period'] = pd.to_datetime(weekly_penalties['period'], errors='coerce').dt.normalize()
solver_weekly_penalties['period'] = pd.to_datetime(solver_weekly_penalties['period'], errors='coerce').dt.normalize()

share_check = (
    metric
    .groupby(['blockchain', 'period'])['weekly_solver_share']
    .sum()
    .rename('share_sum')
    .reset_index()
)

week_check = weekly_penalties.merge(
    share_check,
    on=['blockchain', 'period'],
    how='left'
)

missing_weeks = week_check[
    (week_check['weekly_sum_of_penalty'] > 0)
    & (week_check['share_sum'].isna())
].copy()

if len(missing_weeks):
    print("dropping penalty weeks with no overlapping consistency data:")
    display(missing_weeks[['blockchain', 'period', 'weekly_sum_of_penalty']])

overlap_weeks = week_check[
    (week_check['weekly_sum_of_penalty'] > 0)
    & (week_check['share_sum'].notna())
][['blockchain', 'period']].drop_duplicates()

if len(overlap_weeks) == 0:
    raise RuntimeError("No overlapping chain-weeks between penalties and consistency shares.")

weekly_penalties = weekly_penalties.merge(
    overlap_weeks,
    on=['blockchain', 'period'],
    how='inner'
)

solver_weekly_penalties = solver_weekly_penalties.merge(
    overlap_weeks,
    on=['blockchain', 'period'],
    how='inner'
)

metric = metric.merge(
    overlap_weeks,
    on=['blockchain', 'period'],
    how='inner'
)

share_check = (
    metric
    .groupby(['blockchain', 'period'])['weekly_solver_share']
    .sum()
    .rename('share_sum')
    .reset_index()
)

bad_weeks = weekly_penalties.merge(
    share_check,
    on=['blockchain', 'period'],
    how='left'
)

bad_weeks = bad_weeks[
    (bad_weeks['weekly_sum_of_penalty'] > 0)
    & ~np.isclose(bad_weeks['share_sum'], 1.0, atol=1e-8)
]

if len(bad_weeks):
    print("overlapping penalty weeks with consistency shares not summing to 1")
    display(bad_weeks)
    raise RuntimeError("Cannot allocate all penalty dollars for at least one overlapping chain-week")

weekly_reimbursement = weekly_penalties.merge(
    metric,
    on=['blockchain', 'period'],
    how='left'
)

weekly_reimbursement['weekly_solver_share'] = weekly_reimbursement['weekly_solver_share'].fillna(0)

weekly_reimbursement['penalty_reimbursement_usd'] = (
    weekly_reimbursement['weekly_solver_share']
    * weekly_reimbursement['weekly_sum_of_penalty']
)

reimbursement_by_solver = (
    weekly_reimbursement
    .dropna(subset=['solver_addr'])
    .groupby(['blockchain', 'solver_addr'])
    .agg(
        solver_label=('solver_label', 'first'),
        penalty_reimbursement_usd=('penalty_reimbursement_usd', 'sum'),
        avg_weekly_solver_share=('weekly_solver_share', 'mean'),
    )
    .reset_index()
)

penalties_by_solver = (
    solver_weekly_penalties
    .groupby(['blockchain', 'solver_addr'])
    .agg(
        solver_name=('solver_name', 'first'),
        penalties_paid_usd=('penalties_paid_usd', 'sum'),
        reverts=('reverts', 'sum'),
    )
    .reset_index()
)

penalty_redistribution = penalties_by_solver.merge(
    reimbursement_by_solver,
    on=['blockchain', 'solver_addr'],
    how='outer'
)

penalty_redistribution['penalties_paid_usd'] = penalty_redistribution['penalties_paid_usd'].fillna(0)
penalty_redistribution['penalty_reimbursement_usd'] = penalty_redistribution['penalty_reimbursement_usd'].fillna(0)
penalty_redistribution['reverts'] = penalty_redistribution['reverts'].fillna(0)
penalty_redistribution['avg_weekly_solver_share'] = penalty_redistribution['avg_weekly_solver_share'].fillna(0)

penalty_redistribution['solver_label'] = (
    penalty_redistribution['solver_name']
    .where(
        penalty_redistribution['solver_name'].notna(),
        penalty_redistribution['solver_label']
    )
    .where(
        lambda s: s.notna(),
        penalty_redistribution['solver_addr'].str.slice(0, 10)
    )
    .map(canonical_solver_label_11)
)

penalty_redistribution['net_penalty_pnl_impact_usd'] = (
    penalty_redistribution['penalty_reimbursement_usd']
    - penalty_redistribution['penalties_paid_usd']
)

weekly_share_by_name = weekly_reimbursement.dropna(subset=['solver_addr']).copy()

weekly_share_by_name['solver_label'] = (
    weekly_share_by_name['solver_label']
    .where(
        weekly_share_by_name['solver_label'].notna(),
        weekly_share_by_name['solver_addr'].astype(str).str.slice(0, 10)
    )
    .map(canonical_solver_label_11)
    .astype(str)
)

weekly_share_by_name = (
    weekly_share_by_name
    .groupby(['blockchain', 'period', 'solver_label'], as_index=False)
    .agg(weekly_solver_share=('weekly_solver_share', 'sum'))
)

avg_share_by_name = (
    weekly_share_by_name
    .groupby(['blockchain', 'solver_label'], as_index=False)
    .agg(avg_weekly_solver_share=('weekly_solver_share', 'mean'))
)

penalty_redistribution_by_name = (
    penalty_redistribution
    .copy()
    .assign(
        solver_label=lambda d: d['solver_label']
        .map(canonical_solver_label_11)
        .astype(str)
    )
    .groupby(['blockchain', 'solver_label'], as_index=False)
    .agg(
        penalties_paid_usd=('penalties_paid_usd', 'sum'),
        penalty_reimbursement_usd=('penalty_reimbursement_usd', 'sum'),
        reverts=('reverts', 'sum'),
    )
)

penalty_redistribution_by_name = penalty_redistribution_by_name.merge(
    avg_share_by_name,
    on=['blockchain', 'solver_label'],
    how='left'
)

penalty_redistribution_by_name['avg_weekly_solver_share'] = (
    penalty_redistribution_by_name['avg_weekly_solver_share'].fillna(0)
)

penalty_redistribution_by_name['net_penalty_pnl_impact_usd'] = (
    penalty_redistribution_by_name['penalty_reimbursement_usd']
    - penalty_redistribution_by_name['penalties_paid_usd']
)

penalty_redistribution_by_name = penalty_redistribution_by_name[
    (penalty_redistribution_by_name['penalties_paid_usd'].abs() > 1e-9)
    | (penalty_redistribution_by_name['penalty_reimbursement_usd'].abs() > 1e-9)
].copy()

penalty_redistribution_by_name['plot_size'] = (
    penalty_redistribution_by_name['penalties_paid_usd'].abs()
    + penalty_redistribution_by_name['penalty_reimbursement_usd'].abs()
)

penalty_redistribution_by_name = penalty_redistribution_by_name.sort_values(
    ['blockchain', 'plot_size'],
    ascending=[True, False]
).copy()

if ANONYMIZE_SOLVERS_11:
    penalty_redistribution_by_name['display_solver'] = (
        'Solver '
        + (
            penalty_redistribution_by_name
            .groupby('blockchain')
            .cumcount()
            .add(1)
            .astype(str)
            .str.zfill(2)
        )
    )
else:
    penalty_redistribution_by_name['display_solver'] = penalty_redistribution_by_name['solver_label']

display(
    penalty_redistribution_by_name[
        [
            'blockchain',
            'display_solver',
            'penalties_paid_usd',
            'penalty_reimbursement_usd',
            'net_penalty_pnl_impact_usd',
            'avg_weekly_solver_share',
            'reverts',
        ]
    ]
    .sort_values(['blockchain', 'penalties_paid_usd'], ascending=[True, False])
    .round(4)
)

closure = (
    penalty_redistribution_by_name
    .groupby('blockchain')
    .agg(
        penalties_paid_usd=('penalties_paid_usd', 'sum'),
        penalty_reimbursement_usd=('penalty_reimbursement_usd', 'sum'),
        net_sum_usd=('net_penalty_pnl_impact_usd', 'sum'),
    )
    .reset_index()
)

closure['gap_usd'] = closure['penalty_reimbursement_usd'] - closure['penalties_paid_usd']

closure['gap_bps_of_penalties'] = (
    closure['gap_usd'].abs()
    / closure['penalties_paid_usd'].replace(0, np.nan)
    * 1e4
)

display(closure.round(6))

for ch, d in penalty_redistribution_by_name.groupby('blockchain'):
    d = (
        d.sort_values('plot_size', ascending=False)
         .head(TOP_N_SOLVERS_11)
         .copy()
    )

    if d.empty:
        print(f"no penalty/reimbursement data to plot for {ch}")
        continue

    fig = go.Figure()

    fig.add_bar(
        x=d['display_solver'],
        y=d['penalties_paid_usd'],
        name='penalties paid',
        marker_color='red',
    )

    fig.add_bar(
        x=d['display_solver'],
        y=d['penalty_reimbursement_usd'],
        name='reimbursement',
        marker_color='green',
    )

    fig.update_layout(
        title=f'Penalty P&L redistribution by solver — {ch}',
        xaxis_title='solver',
        yaxis_title='USD',
        barmode='group',
        width=max(1000, 45 * len(d) + 350),
        height=650,
        legend_title_text='',
    )

    fig.update_xaxes(tickangle=-45)
    fig.show()

## 13. Ethereum not-settled attempts vs Binance price risk
Compares winning attempts Binance price movements during the 26-second exclusivity window (ethereum).
Tests whether non-settlement was economically consistent with adverse price movement, including whether the estimated settlement loss exceeded the applicable penalty.

In [ ]:
START, END = pd.Timestamp("2026-06-01", tz="UTC"), pd.Timestamp("2026-06-30", tz="UTC")
T_EXCL = 26.6
BINANCE_CACHE = Path("../data/binance_ticks"); BINANCE_CACHE.mkdir(parents=True, exist_ok=True)

WETH = "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"
USDC = "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"
PAIRS = {"WETH/USDC": (WETH, USDC, "ETHUSDT")}

KEY = ["blockchain", "auction_id", "solver"]

COW_CSV = Path("../data/ethereum_2026-06-01_2026-06-30.csv")
REPO_ROOT = Path("..").resolve()

if not COW_CSV.exists():
    load_dotenv(REPO_ROOT / ".env", override=True)
    subprocess.run([
        "uv", "run", "python", "scripts/fetch_penalties_data.py",
        "--chain", "ethereum",
        "--start", "2026-06-01",
        "--end", "2026-06-30",
        "--db-timeout", "1800",
    ], cwd=REPO_ROOT, check=True)

raw = prepare(pd.read_csv(COW_CSV, low_memory=False))
raw["auction_timestamp"] = pd.to_datetime(raw["auction_timestamp"], utc=True, errors="coerce")
raw = raw[raw.blockchain.eq("ethereum") & raw.auction_timestamp.ge(START) & raw.auction_timestamp.lt(END)]

order_counts = raw.groupby(KEY)["order_uid"].nunique().rename("n_orders").reset_index()
df, funnel = apply_filters(raw, fok_only=True, in_market_only=True, exclude_penalty_excluded=False)

df["sell"] = df.sell_token.astype(str).str.lower()
df["buy"] = df.buy_token.astype(str).str.lower()
df["pair"] = None
for name, (base, quote, _) in PAIRS.items():
    df.loc[(df.sell.eq(base) & df.buy.eq(quote)) | (df.sell.eq(quote) & df.buy.eq(base)), "pair"] = name

ATTEMPTS = (df[df.pair.notna()].merge(order_counts, on=KEY, how="left")
            .query("n_orders == 1").drop_duplicates(KEY).copy())

print("Funnel:", funnel, "| attempts:", len(ATTEMPTS))
display(ATTEMPTS.groupby("pair").agg(
    attempts=("auction_id", "size"), not_settled=("reverted", "sum"),
    not_settled_rate=("reverted", "mean"), median_size_usd=("size_usd", "median")).round(4))

def load_symbol(symbol):
    cols = ["id", "price", "qty", "first", "last", "ts", "maker", "best"]
    days = []
    for day in pd.date_range("2026-06-01", "2026-06-30"):
        d = day.strftime("%Y-%m-%d")
        path = BINANCE_CACHE / f"{symbol}-aggTrades-{d}.zip"
        if not path.exists():
            print("Downloading:", symbol, d)
            urllib.request.urlretrieve(
                f"https://data.binance.vision/data/spot/daily/aggTrades/{symbol}/{path.name}", path)
        with zipfile.ZipFile(path) as z, z.open(z.namelist()[0]) as f:
            t = pd.read_csv(f, header=None, names=cols, usecols=["price", "ts"])
        t = t.assign(price=pd.to_numeric(t.price, errors="coerce"),
                     ts=pd.to_numeric(t.ts, errors="coerce")).dropna().query("price > 0")
        unit = "us" if t.ts.iloc[0] > 10**14 else "ms"
        t["time"] = pd.to_datetime(t.ts.astype("int64"), unit=unit, utc=True)
        days.append(t[["time", "price"]])
    ticks = pd.concat(days).sort_values("time").drop_duplicates("time", keep="last").set_index("time")["price"]
    return ticks.resample("1s", label="right", closed="right").last().ffill(limit=5)

symbols = {s: load_symbol(s) for s in ["ETHUSDT", "USDCUSDT"]}
ETH_USDC = symbols["ETHUSDT"] / symbols["USDCUSDT"]

for col in ["executed_sell", "executed_buy"]:
    ATTEMPTS[col] = pd.to_numeric(ATTEMPTS[col], errors="coerce")

ATTEMPTS["weth_qty"] = np.where(
    ATTEMPTS["sell"].eq(WETH),
    ATTEMPTS["executed_sell"] / 1e18,
    ATTEMPTS["executed_buy"] / 1e18,
)

ATTEMPTS["usdc_qty"] = np.where(
    ATTEMPTS["sell"].eq(USDC),
    ATTEMPTS["executed_sell"] / 1e6,
    ATTEMPTS["executed_buy"] / 1e6,
)

ATTEMPTS["execution_price"] = (
    ATTEMPTS["usdc_qty"] / ATTEMPTS["weth_qty"]
)

ATTEMPTS = ATTEMPTS[
    ATTEMPTS["weth_qty"].gt(0)
    & ATTEMPTS["usdc_qty"].gt(0)
].copy()

def market_features(attempts):
    rows = []

    for r in attempts.itertuples():
        deadline = r.auction_timestamp
        start = deadline - pd.Timedelta(seconds=T_EXCL)
        start_second = start.ceil("s")
        deadline_second = deadline.floor("s")
        window_seconds = pd.date_range(
            start=start_second,
            end=deadline_second,
            freq="1s",
        )
        path = ETH_USDC.reindex(window_seconds)
        if (
            len(path) < 2
            or pd.isna(path.iloc[0])
            or pd.isna(path.iloc[-1])
        ):
            rows.append({
                "estimated_window_start": start,
                "start_market_price": np.nan,
                "deadline_market_price": np.nan,
                "start_pnl_usdc": np.nan,
                "deadline_pnl_usdc": np.nan,
                "worst_pnl_usdc": np.nan,
                "deadline_loss_usdc": np.nan,
                "max_loss_usdc": np.nan,
                "penalty_usdc": np.nan,
            })
            continue
        
        path = path.dropna()

        if r.sell == WETH:
            pnl = r.weth_qty * (path - r.execution_price)
        else:
            pnl = r.weth_qty * (r.execution_price - path)

        start_price = float(path.iloc[0])
        deadline_price = float(path.iloc[-1])

        rows.append({
            "estimated_window_start": start,
            "start_market_price": start_price,
            "deadline_market_price": deadline_price,
            "start_pnl_usdc": float(pnl.iloc[0]),
            "deadline_pnl_usdc": float(pnl.iloc[-1]),
            "worst_pnl_usdc": float(pnl.min()),
            "deadline_loss_usdc": max(0, -float(pnl.iloc[-1])),
            "max_loss_usdc": max(0, -float(pnl.min())),
            "penalty_usdc": (
                max(float(r.penalty_native), 0)
                / 1e18
                * deadline_price
            ),
        })

    out = pd.concat(
        [attempts, pd.DataFrame(rows, index=attempts.index)],
        axis=1,
    )

    out["not_settled"] = out["reverted"].fillna(False)

    out["became_loss_making_by_deadline"] = (
        out["start_pnl_usdc"].ge(0)
        & out["deadline_pnl_usdc"].lt(0)
    )

    out["became_loss_making_during_window"] = (
        out["start_pnl_usdc"].ge(0)
        & out["worst_pnl_usdc"].lt(0)
    )

    out["price_consistent_not_to_settle"] = (
        out["not_settled"]
        & out["became_loss_making_by_deadline"]
        & out["deadline_loss_usdc"].gt(out["penalty_usdc"])
    )

    return out

SECTION13 = market_features(ATTEMPTS)
total_revert_rate = SECTION13["not_settled"].mean()
price_sensitive_revert_rate = (
    SECTION13["price_consistent_not_to_settle"].sum()
    / len(SECTION13)
)
residual_non_price_revert_rate = (
    total_revert_rate - price_sensitive_revert_rate
)
price_consistent_share_of_reverts = (
    SECTION13.loc[
        SECTION13["not_settled"],
        "price_consistent_not_to_settle",
    ].mean()
)

print(
    f"Total revert rate: {total_revert_rate:.2%}\n"
    f"Price-sensitive revert rate: {price_sensitive_revert_rate:.2%}\n"
    f"Residual non-price revert rate: {residual_non_price_revert_rate:.2%}\n"
    f"Price-consistent share of reverts: "
    f"{price_consistent_share_of_reverts:.2%}"
)
ALL_COVERED = SECTION13.dropna(
    subset=[
        "start_pnl_usdc",
        "deadline_pnl_usdc",
        "worst_pnl_usdc",
    ]
).copy()
ALL_COVERED["profitability_state"] = np.select(
    [
        ALL_COVERED["start_pnl_usdc"].lt(0),

        ALL_COVERED["start_pnl_usdc"].ge(0)
        & ALL_COVERED["deadline_pnl_usdc"].lt(0),

        ALL_COVERED["start_pnl_usdc"].ge(0)
        & ALL_COVERED["worst_pnl_usdc"].lt(0)
        & ALL_COVERED["deadline_pnl_usdc"].ge(0),

        ALL_COVERED["start_pnl_usdc"].ge(0)
        & ALL_COVERED["worst_pnl_usdc"].ge(0),
    ],
    [
        "Already loss-making at start",
        "Became loss-making and remained so at deadline",
        "Temporarily loss-making but recovered",
        "Profitable throughout window",
    ],
    default="Other",
)
revert_by_state = (
    ALL_COVERED
    .groupby("profitability_state")
    .agg(
        attempts=("auction_id", "size"),
        settled=("not_settled", lambda x: (~x).sum()),
        not_settled=("not_settled", "sum"),
        not_settled_rate=("not_settled", "mean"),
        median_start_pnl_usdc=("start_pnl_usdc", "median"),
        median_deadline_pnl_usdc=("deadline_pnl_usdc", "median"),
    )
    .reset_index()
)

display(revert_by_state.round(4))

BECAME_LOSS_MAKING = ALL_COVERED[
    ALL_COVERED["became_loss_making_during_window"]
].copy()

print({
    "attempts": len(BECAME_LOSS_MAKING),
    "settled": (~BECAME_LOSS_MAKING["not_settled"]).sum(),
    "not_settled": BECAME_LOSS_MAKING["not_settled"].sum(),
    "not_settled_rate": BECAME_LOSS_MAKING["not_settled"].mean(),
})

NOT_SETTLED = SECTION13[
    SECTION13["not_settled"]
].copy()

COVERED = NOT_SETTLED.dropna(
    subset=[
        "start_market_price",
        "deadline_market_price",
        "penalty_usdc",
    ]
)

result = pd.DataFrame({
    "metric": [
        "Total not-settled attempts",
        "With usable Binance prices",
        "Profitable at start, loss-making at deadline",
        "Deadline loss greater than penalty",
        "Temporarily became loss-making during window",
        "Already loss-making at start",
    ],
    "attempts": [
        len(NOT_SETTLED),
        len(COVERED),
        COVERED["became_loss_making_by_deadline"].sum(),
        COVERED["price_consistent_not_to_settle"].sum(),
        COVERED["became_loss_making_during_window"].sum(),
        COVERED["start_pnl_usdc"].lt(0).sum(),
    ],
})

result["share_of_not_settled"] = (
    result["attempts"] / len(NOT_SETTLED)
)

display(result.round(4))

plot_data = result.iloc[[2, 3, 4]].copy()
plot_data["percentage"] = plot_data["share_of_not_settled"] * 100

fig = go.Figure(go.Bar(
    x=plot_data["metric"],
    y=plot_data["percentage"],
    text=[f"{p:.1f}%<br>(n={n})"
          for p, n in zip(plot_data["percentage"], plot_data["attempts"])],
    textposition="outside",
    marker_color="#4f81bd",
))

fig.update_layout(
    title="Non-settlements consistent with adverse price movement",
    yaxis_title="Share of not-settled attempts (%)",
    xaxis_title="",
    yaxis_range=[0, plot_data["percentage"].max() * 1.25],
    width=850, height=500,
    template="simple_white",
    margin=dict(t=60, b=90),
)
fig.update_xaxes(tickangle=-15)
fig.show()

details = NOT_SETTLED[
    [
        "auction_timestamp",
        "estimated_window_start",
        "auction_id",
        "solver_name",
        "kind",
        "weth_qty",
        "execution_price",
        "start_market_price",
        "deadline_market_price",
        "start_pnl_usdc",
        "deadline_pnl_usdc",
        "deadline_loss_usdc",
        "penalty_usdc",
        "became_loss_making_by_deadline",
        "price_consistent_not_to_settle",
    ]
].sort_values(
    "deadline_loss_usdc",
    ascending=False,
)

numeric_cols = [
    "weth_qty",
    "execution_price",
    "start_market_price",
    "deadline_market_price",
    "start_pnl_usdc",
    "deadline_pnl_usdc",
    "deadline_loss_usdc",
    "penalty_usdc",
]

details[numeric_cols] = details[numeric_cols].round(4)
display(details)